<a href="https://colab.research.google.com/github/ribesstefano/PROTAC-Splitter/blob/main/notebooks/trl_protac_splitter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finetuning PROTAC-Splitter in TRL

## Setup

In [14]:
texts = """['CC(=O)NC(C(=O)N1CC(O)CC1C(=O)NC(CC(=O)N1CCC(C#Cc2ccc(C(=O)NC3C(C)(C)C(Oc4ccc(C#N)c(Cl)c4)C3(C)C)cc2)CC1)c1ccccc1)C(C)C<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]CCOCCOCCOCCOCCOCCNC(=O)CCC[*:1].[*:1]N1CCC2CCC(NC(=O)NC(CCC(Oc3ccc(C)(C)c(F)cc3F)c(Nc4ccccc4S(=O)(=O)C(N)=O)c3)ccc(-c2)cc1</s>', 'CCC(C(=O)N1CCCCC1C(=O)OC(CCc1ccc(OC)c(OC)c1)c1ccccc1OCC(=O)NCCCOCCOCCOCCCNC(=O)COc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O)c1cc(OC)c(OC)c(OC)c1<s><s>[*:1]N1CCN(Cc2ccc(C(=O)Nc3ccc(C)c(C#Cc4ccc4[nH]ncc5c4)c3)cc2C(F)(F)F)CC1.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)NCCOCCOCCOCCOCCOCCOCCOCCOCCOCCOCC[*:1]</s>', 'CC1Cc2c([nH]c3ccccc23)C(c2cnc(N3CCC4(CC3)CC(CN3CC5(CCN(c6ccc7c(c6)CN(C6CCC(=O)NC6=O)C7=O)CC5)C3)C4)nc2)N1CC(C)(C)F<s><s>[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]CCCCCCCCCCNC(=O)[*:1].[*:1]Nc1cc(C(=O)Nc2ccccc2N(C1)C1CC(C)(C)C)C.[*:2]CCCCCCCCCCNC(=O)[*:1]</s>', 'CCCS(=O)(=O)Nc1ccc(F)c(-n2cc(-c3cncnc3)c3nc(N(C)C4CCN(C(=O)CCOCCOCCC(=O)NC(C(=O)N5CC(O)CC5C(=O)NCc5ccc(-c6scnc6C)cc5)C(C)(C)C)CC4)ccc32)c1F<s><s>[*:2]c1ccc2c(c1)sc1nc(-c3ccc(NC(=O)Nc4cc(C(C)(C)C)on4)cc3)cn12.[*:2]c1ccc2c(c1)CN(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCn1cc(CC)nn1)c2ccc(OC)cc2)c2c1)CC1</s>', 'COC(=O)CC1N=C(c2ccc(Cl)cc2)c2c(sc(C(=O)NCCCCCCCCOc3ccc4c(c3)C(=O)N(C3CCC(=O)NC3=O)C4=O)c2C)-n2c(C)nnc21<s><s>[*:1]c1ccc(Nc2ncc(Br)c(NCCCN(C)C(=O)C3CCC3)n2)cc1.[*:2]NC(C(=O)N1CC(O)CC1C(=O)NC(C)c1ccc(-c2scnc2C)cc1)C(C)(C)C.[*:2]C(=O)N1CCC(N2CCC(C#[*:1])C[*:1])CC2)C[*:1</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)COCC(F)(F)C(F)(F)C(F)(F)COCCCOc2ccc(N3C(=S)N(c4ccc(C#N)c(C(F)(F)F)c4)C(=O)C3(C)C)cc2)C(C)(C)C)cc1<s><s>[*:1]c1nc(-c2cn(C)c(=O)c3cc(C(=N)NC4CCS(=O)(=O)CC4)sc23)cc1OC.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)NCCOCCOCCOCCOCCn1cc(C)c1ccc(-n2)cc(C)c(-n2c2c(C)cc1</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)COCCOCCOCCOCCOCCOCCOc2ccc(C=NNC(=O)c3c(-c4ccc(O)cc4)sc4cc(O)ccc34)cc2)C(C)(C)C)cc1<s><s>[*:1]c1cc2c(-c3cc(NS(=O)(=O)CC)cc3c(C)cc(F)c(Cl)c3)c2c1.[*:2]c1ccc2c(c1)C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]N1CCC(C)CCN2CCN[*:1])CC2)CC1</s>', 'Cc1cc(C(C(=O)N2CC(O)CC2C(=O)NC(CC(=O)N2CCC(N3CC(c4ccc(C(=O)NC5C(C)(C)C(Oc6ccc(C#N)c(Cl)c6)C5(C)C)cc4)C3)CC2)c2ccccc2)C(C)C)on1<s><s>[*:2]Nc1cccc2c1CN(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)COCCOCCOCCn1cc(CCC[*:1])nn1.[*:1]CNC(=O)c1ccc(C)c(-n2c3ccc(Oc4cc(F)ccc4Cl)cc3)c(Br)c2=O)c1</s>', 'COc1cc(-c2cn(C)c(=O)c3cnccc23)c(OC)cc1CN1CC(NC(=O)COCCOCCOc2cc(-c3scnc3C)ccc2CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)C2(F)CC2)C(C)(C)C)C1<s><s>[*:1]c1ccc(C(=O)NC2C(C)(C)C(Oc3ccc(C#N)c(Cl)c3)C2(C)C)cc1.[*:2]c1ccc(Cl)cc2)C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]N1CCC(CN2CCN([*:1])CC2)CC1</s>', 'Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C(=O)CNc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)CC2)n1<s><s>[*:1]c1ccc(C(=O)NC2C(C)(C)C(Oc3ccc(C#N)c(Cl)c3)C2(C)C)cc1.[*:2]C(NC(=O)C1CC(O)CN1C(=O)C(c1cc(C)no1)C(C)C)c1ccc(-c2scnc2C)cc1.[*:2]CC(=O)N1CCC(N2CCC(C#C[*:1])CC2)CC1</s>', 'Cc1[nH]c(C=C2C(=O)Nc3ccc(F)cc32)c(C)c1C(=O)NCCCCCCCCNC(=O)c1ccc2c(c1)C(=O)N(C1CCC(=O)NC1=O)C2=O<s><s>[*:1]c1cc(C=CC(=O)Nc2nccc(C(C)(C)C)c(OC)cc2C)c1cc(-c2scnc2C)ccc1CNC(=O)C1CC(O)CN1C(=O)C(NC(C)=O)C(C)(C)C.[*:2]C(=O)O)O.[*:1]OCCOCCOCCOCCOCC[*:1]</s>', 'COc1ccc(-c2ocnc2C(=O)NCCCc2cn(CCOCCOCCOCCOCCNc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)nn2)cc1I<s><s>[*:1]CNC(=O)c1cc(OC)c(-n2c(C)cc(OCc3ccc(F)cc3F)c(Br)c2=O)c1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCCNC(=O)C[*:1]</s>', 'CNC(=O)c1sc(-c2ccnc(NC(=O)C3CC3)c2)nc1OCCOCCOCCOCCOCCOc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O<s><s>[*:1]Nc1cccc(C(CCc2cc(OC)c(OC)c2)C(=O)C2N(C(=O)C(C)C)c2ccc(OC)c(OC)c1.[*:2]NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)cc1)C(C)(C)C.[*:2]C(=O)COCCOCCn1cc(C[*:1])nn1</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)COCCOCCOCCOCCOCc2cn(CCNC(=O)CC3N=C(c4ccc(Cl)cc4)c4c(sc(C)c4C)-n4c(C)nnc43)nn2)C(C)(C)C)cc1<s><s>[*:1]N1CCN(c2ccc(Nc3ncc4c(C)c(C(C)=O)c(=O)n(C5CCCC5)c4n3)nc2)CC1.[*:2]NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)cc1)C(C)(C)C.[*:2]C(=O)CCC(=O)N(C)CCCCCC)C[*:1]</s>', 'NC(=O)CCC(NC(=O)C1CCC2CCN(C(=O)CCCCC#Cc3cccc4c3CN(C3CCC(=O)NC3=O)C4=O)CC(NC(=O)c3cc4cc(C(F)(F)P(=O)(O)O)ccc4[nH]3)C(=O)N21)C(=O)NC(c1ccccc1)c1ccccc1<s><s>[*:1]c1ccc(C(=O)NC2C(C)(C)C(Oc3ccc(C#N)c(Cl)c3)C2(C)C)cc1.[*:2]C(NC(=O)C1CC(O)CN1C(=O)C(NC(C)=O)C(C)(C)C)c1ccc(-c2scnc2C)cc1.[*:2]CCCCCCCCCC#NN)C(N)C[*:1]</s>', 'CN(c1nccnc1CNc1nc(Nc2ccc(NC(=O)c3cn(CCOCCOCCNc4cccc5c4C(=O)N(C4CCC(=O)NC4=O)C5=O)nn3)cc2)ncc1C(F)(F)F)S(C)(=O)=O<s><s>[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]CC(=O)NCCOCCOCCN[*:1]</s>', 'COc1cc(-c2cn(C)c(=O)c3cnccc23)cc(OC)c1CN1CCN(CCOCCOc2cc(-c3scnc3C)ccc2CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)C2(F)CC2)C(C)(C)C)CC1<s><s>[*:1]#Cc1ccc(C(=O)NC2C(C)(C)C(Oc3ccc(C#N)c(Cl)c3)C2(C)C)cc1.[*:2]CC(NC(=O)C1CC(O)CN1C(=O)C(NC(C)=O)C(C)(C)C)c1ccc(-c2scnc2C)cc1.[*:2]C(=O)N1CCC(N2CCC(C#[*:1])CC2)CC1</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCCCCCCOc2ccc(C=NNC(=O)c3cccc(Oc4c(-c5ccc(O)cc5)sc5cc(O)ccc45)c3)cc2)C(C)(C)C)cc1<s><s>[*:1]#Cc1ccc(N2C(=S)N(c3cnc(C#N)c(C(F)(F)F)c3)C(=O)C23CCC3)cc1.[*:2]C(NC(=O)C1CC(O)CN1C(=O)C(c1cc(C)no1)C(C)C)c1ccc(-c2scnc2C)cc1.[*:2]CC(=O)N1CCC(N2CCC(C#C#N)CC2)CC1)C1</s>', 'Nc1c(-c2nc3cc(N4CCN(C(=O)CCCCC#Cc5cccc6c5C(=O)N(C5CCC(=O)NC5=O)C6=O)CC4)ccc3[nH]2)c(=O)[nH]c2cccc(F)c12<s><s>[*:1]c1ccc2c(c1)sc1nc(-c3ccc(NC(=O)Nc4cc(C(C)(C)C)on4)cc3)cn12.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)NCCCCNC(=O)C1CC(N2CCN(O)CC(O)C(=O)C(C)c3ccc(-c3ccc([*:2])cc1</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)COCCCOCCNC(=O)CC2N=C(c3ccc(Cl)cc3)c3c(sc(C)c3C)-n3c(C)nnc32)C(C)(C)C)cc1<s><s>[*:1]OC1CCC(CC(C)C2CC(=O)C(C)C=C(C)C(O)C(OC)C(=O)C(C)CC(C)C=CC=CC=C(C)C(OC)CC3CCC(C)C(O)(O3)C(=O)C(=O)N3CCCCC3C(=O)O2)CC1OC.[*:2]c1cccc2c1OC</s>', 'Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(CCCCCCOCCCCCOCCCCCC(=O)NC(C(=O)N3CC(O)CC3C(=O)NCc3ccc(-c4scnc4C)cc3)C(C)(C)C)CC2)n1<s><s>[*:1]CN1CCN(c2cc(Cl)cc3c2nc(N2CC4(CNC4)C2)n3Cc2cc(C)c(F)c(C)c2)CC1.[*:2]CC(=O)NC(C(=O)N1CC(O)CC1C(=O)NC(C)c1ccc(-c2scnc2C)cc1)C(C)(C)C.[*:2]N1CCC(CN2CCN(C(=O)C[*:1])CC2)CC1</s>', 'Nc1ncnc2c1c(-c1ccc(Cl)cc1)nn2CCCNc1ccc(OCCOCCOCCNc2cccc3c2C(=O)N(C2CCC(=O)NC2=O)C3=O)cc1<s><s>[*:1]Nc1nc(N2CCN(C(=O)C(Cc3ccc(O)cc3)n3cc(C(N)C(C)CC)nn3)CC2)nc(N2CCN(C(=O)C(Cc3ccc(O)cc3)n3cc(C(N)C(C)CC)nn3)CC2)n1.[*:2]N1CCN(C(=O)N2C(c3ccc(OC)cc3OC(C)C)=NC(c3OC)C2c3OC)cc2c2c2ccc(Cl)CC1=O.[*:2]CC(=O)NCCOCCOCCOCCOCCOCCn1cc(COCCOCCOCC[*:1])nn1</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)COCCNc2ncc(C(=O)Nc3ccc(OC(F)(F)Cl)cc3)cc2-c2ccn[nH]2)C(C)(C)C)cc1<s><s>[*:1]OC(=O)C(C)C1CC(Oc2ccc(C#N)c(Cl)c2OC)C1.[*:2]C(=O)NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)cc1)C(C)(C)C.[*:2]C(=O)CCCCCCCCCCC#C[*:1]</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCOCCOCCOCCOCCNCCCONC(=O)c2ccc(F)c(F)c2Nc2ccc(I)cc2F)C(C)(C)C)cc1<s><s>[*:1]N1CCCC(n2nc(-c3ccc(Oc4ccccc4)cc3)c3c(N)ncnc32)C1.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)NCCOCCOCCNC(=O)c1ccc(NC(=O)c1)cc(C(=O)O)C(C)(C)C=O.[*:2]OCCOCCOCCn1cc(CCC(=O)[*:1])nn1</s>', 'O=C(CCCCCNC(=O)CCCCC(=O)NCCN1C(=O)c2cccc3c(Sc4ccc(Br)cc4)ccc(c23)C1=O)NCCCCCCNc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O<s><s>[*:1]N1CCN(c2ccc(Nc3ncc4c(C)c(C(C)=O)c(=O)n(C5CCCC5)c4n3)nc2)CC1.[*:2]CC(=O)NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)cc1)C(C)(C)C.[*:2]C(=O)CCOCCOCCOCC[*:1]</s>', 'CN(CCCCCCNc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O)CCC(CSc1ccccc1)Nc1ccc(S(=O)(=O)NC(=O)c2ccc(N3CCN(CC4=C(c5ccc(Cl)cc5)CCC(C)(C)C4)CC3)cc2)cc1S(=O)(=O)C(F)(F)F<s><s>[*:1]N1CCN(Cc2ccc(C(=O)Nc3ccc(C)c(Nc4nccc(-c5cccnc5)n4)c3)cc2)CC1.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)NCCCCCCCCNC(=O)CCC(=O)[*:1]</s>', 'CCCNNC(=O)c1ccc(-c2ccc(NC(=O)COCCOCCOCCNc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)cc2)cc1<s><s>[*:1]N1CCN(c2ccc(Nc3ncc4c(C)c(C(C)=O)c(=O)n(C5CCCC5)c4n3)nc2)CC1.[*:2]NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)cc1)C(C)(C)C.[*:2]C(=O)CCOCCOCCOCCn1cc(C[*:1])nn1</s>', 'Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(CCCCNc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)CC2)n1<s><s>[*:1]c1ccc(N(C)C)c(OC)c(Nc2ccc(C#N)c(C(F)(F)F)c2)cc1.[*:2]COc1ccc2c(c1)C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]N1CCN(CC(N2CCN(C(=O)C(c3ccc(OC)cc3ccc(C)C3OC)C3CCN(C)C3CCN(C(=O)C)C3CCN([*:1])CC3)CC3)nn2)C1</s>', 'COc1cc(N2CCC(OCC(=O)NC(C(=O)N3CC(O)CC3C(=O)NC(C)c3ccc(-c4scnc4C)cc3)C(C)(C)C)CC2)ccc1NC(=O)c1cccc(-c2ccn[nH]2)n1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC(=O)C[*:1].[*:1]N1CCC(N2CCN(C(=O)Nc3ncc(C(C#N)c(Cl)c4ccccc4S(=O)(=O)C(C)n3)cc2C)CC1</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCCCCCOc2ccc(C=NNC(=O)c3ccc(C(=O)c4c(-c5ccc(O)cc5)sc5cc(O)ccc45)cc3)cc2)C(C)(C)C)cc1<s><s>[*:1]N1CCN(CCC(CSc2ccccc2)Nc2ccc(S(=O)(=O)NC(=O)c3ccc(N4CCN(CC5=C(c6ccc(Cl)cc6)CCC(C)(C)C5)CC4)cc3)cc2S(=O)(=O)C(F)(F)F)CC1.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)NCCCCNC(=O)CCC(C[*:1]</s>', 'COc1cc(N=Nc2ccc3c(c2)C(=O)N(C2CCC(=O)NC2=O)C3=O)cc(OC)c1OCC(=O)NCCNC(=O)CC1N=C(c2ccc(Cl)cc2)c2c(sc(C)c2C)-n2c(C)nnc21<s><s>[*:1]N1CCN(C2CCN(c3ccc(Nc4ncc(Cl)c(Nc5ccccc5P(C)(C)=O)n4)c(OC)c3)CC2)CC1.[*:2]CC(=O)NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)cc1)C(C)(C)C.[*:2]C(=O)N1CCC(=O)NCCOCCOCC[*:1]</s>', 'CNC(C)C(=O)NC(C(=O)N1CCCC1c1nc(C(=O)c2ccc(F)cc2)cs1)C1CCN(C(=O)c2cnc(N3CCC(CCCOc4cc5ncnc(Nc6n[nH]c(C)c6C)c5cc4S(=O)(=O)C(C)(C)C)CC3)nc2)CC1<s><s>[*:1]c1ccc(C(=O)NC2C(C)(C)C(Oc3ccc(C#N)c(Cl)c3)C2(C)C)cc1.[*:2]c1ccc2c(c1)C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]N1CCC(CN2CCN(C(=O)[*:1])CC2)C1</s>', 'CC(=O)NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)cc1)C(C)(C)SCCOCCOCCNC(=O)CC1N=C(c2ccc(Cl)cc2)c2c(sc(C)c2C)-n2c(C)nnc21<s><s>[*:1]c1cnc(OCC2NC(=O)C(F)C2CC)c2cc(OC)c(C(N)=O)cc12.[*:2]NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)cc1)C(C)(C)C.[*:2]C(=O)CCOCCOCCOCCNC(=O)C[*:1]</s>', 'Nc1nc(Nc2ccc(CNC(=O)CCCCCNc3cccc4c3CN(C3CCC(=O)NC3=O)C4=O)cc2)nn1-c1cc2c(nn1)-c1ccccc1CCC2<s><s>[*:1]OC1CCC(CC(C)C2CC(=O)C(C)C=C(C)C(O)C(OC)c3ccc(Cl)cc3)CC2)CC1.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)NCCCCNC(=O)CCC(=O)NCCOCCOCCOCCOCCNC(=O)[*:1]</s>', 'Cn1cc(-c2ccccc2Oc2ccccc2)c2cc(C(=O)NCCCCC(=O)NCCCCNc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)[nH]c2c1=O<s><s>[*:1]OC1CCC(NC(=O)c2ccc(Nc3ncc4c(n3)N(CC3CCCCC3)C(CC)c3nnc(C)n3-4)c(OC)c2)CC1.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)NCCOCCOCCOCCOCCOCCNC(=O)CCC(=O)[*:1]</s>', 'COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCN(C(=O)CCOCCOCCOCCOCCOCCC(=O)NC(C(=O)N2CC(O)CC2C(=O)NCc2ccc(-c3scnc3C)cc2)C(C)(C)C)CC1<s><s>[*:1]c1cccc(C(=O)NC2CCC(N(CCC)C(=O)NC(=O)C(NC(=O)c3ccccc3)c(-c4)CCC(C)CC)c2)c1.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)NCCOCCOCCOCCCC(=O)N[*:1]</s>', 'COc1cc2c(Oc3ccc(N(C(=O)C4(C(N)=O)CC4)c4ccc(F)cc4)cc3F)ccnc2cc1OCCCOCCCOCCC(=O)NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)cc1)C(C)(C)C<s><s>[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)NCCCCCCC(=O)CCC(=O)[*:1].[*:1]N1CCC(CCC(N2CCC(N)=O)C(c3ccc(NC(=O)c4ccc(C(C#N)c(Cl)c6)cccc4)c3)cc2C1</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)COCCCOCCCCCOc2ccc(N3C(=S)N(c4ccc(C#N)c(C(F)(F)F)c4)C(=O)C3(C)C)cc2)C(C)(C)C)cc1<s><s>[*:1]N1CCCC(n2nc(-c3ccc(Oc4ccccc4)cc3)c3c(N)ncnc32)C1.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)N1CCC(CN2CCN(C(=O)[*:1])CC2)CC1</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCOCCOCCOCCOCCNC(=O)c2ccc(-c3ccc(N4CCN(C)CC4)c(NC(=O)c4c[nH]c(=O)cc4C(F)(F)F)c3)cc2)C(C)(C)C)cc1<s><s>[*:1]c1ccc2c(c1)sc1nc(-c3ccc(NC(=O)Nc4cc(C(C)(C)C)on4)cc3)cn12.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)N1CCC(CN2CCN(C(=O)c3ccc(NC(=O)cc3=O)C3ccc(C3)cc3)C2=O.[*:2]CC1cc(OCCOCCOCC[*:1])nn1</s>', 'C#Cc1ccc(C(CC(=O)N2CCC(N3CCC(C#Cc4ccc(C(=O)NC5C(C)(C)C(Oc6ccc(C#N)c(Cl)c6)C5(C)C)cc4)CC3)CC2)NC(=O)C2CC(O)CN2C(=O)C(c2cc(C)no2)C(C)C)cc1<s><s>[*:1]c1ccc(-c2cn(CCCC)c(=O)c3cc(C(=N)NC4CCS(=O)(=O)CC4)sc23)cc1OC.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)N1CCC(CCN(C2CCN(C(=O)Nc3ccc(C(=O)NO)cc3)C4c(OC)C3)CC2)CC1</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)COCCOCCOCCOCc2cn(CCNC(=O)CC3N=C(c4ccc(Cl)cc4)c4c(sc(C)c4C)-n4c(C)nnc43)nn2)C(C)(C)C)cc1<s><s>[*:1]N1CCN(CC#Cc2ccc(NC(=O)c3c(C(=O)Nc4cccc(F)c4)c3C(F)(F)F)c2)CC1.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)NCCCCNC(=O)CCC(=O)[*:1]</s>', 'CN1c2ccccc2C(=O)Nc2cnc(Nc3ccc(N4CCN(C(=O)CCOCCNc5cccc6c5C(=O)N(C5CCC(=O)NC5=O)C6=O)CC4)cc3)nc21<s><s>[*:1]N1CCN(CCC(CSc2ccccc2)Nc2ccc(S(=O)(=O)NC(=O)c3ccc(N4CCN(CC5=C(c6ccc(Cl)cc6)CCC(C)(C)C5)CC4)cc3)cc2S(=O)(=O)C(F)(F)F)CC1.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)NCCCCNC(=O)NCCCCNC(=O)CCC(=O)[*:1]</s>', 'COc1ccc(-c2ocnc2C(=O)NCCCc2cn(CCOCCOCCOCCNC(=O)CNc3cccc4c3CN(C3CCC(=O)NC3=O)C4=O)nn2)cc1Cl<s><s>[*:1]c1cccc(-c2c(C(=O)Nc3ccccc3)c(C(C)C)n(CCC(O)CC(O)CC(=O)O)c2-c2ccc(F)cc2)c1.[*:2]c1ccc2c(c1)C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)NCCCCNC(=O)C[*:1]</s>', 'Nc1nc2c(ncn2C2OC(COP(=O)(O)OP(=O)(O)NCCCCCC(=O)Nc3cccc4c3CN(C3CCC(=O)NC3=O)C4=O)C(O)C2O)c(=O)[nH]1<s><s>[*:2]CC(=O)NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)cc1)C(C)(C)C.[*:2]OCCOCCNC(=O)[*:1].[*:1]N1CCC(c2ccc(Nc3ncc4c(C)C(NS(=O)(=O)c5cccc5)C(F)c4)n3)cc2)CC1</s>', 'CCC(=C(c1ccc(O)cc1)c1ccc(OCCN(C)C(=O)COCCOCCOCCOCCNC(=O)C(NC(=O)C2CCCN2C(=O)C(NC(=O)C(C)NC)C2CCCCC2)C(c2ccccc2)c2ccccc2)cc1)c1ccccc1<s><s>[*:1]N1CCN(Cc2ccc(NC(=O)c3[nH]ncc3Nc3ncnc4[nH]ccc34)cc2)CC1.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)NCCOCCOCCOCCOCCNC(=O)c1ccc(C[*:1])cc1</s>', 'CCn1c(-c2nonc2N)nc2c(C#CC(C)(C)O)ncc(OCCCNCCCC(=O)NCCOCCOCCOCCOCCOCCNc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)c21<s><s>[*:1]N1CCN(CCC(CSc2ccccc2)Nc2ccc(S(=O)(=O)NC(=O)c3ccc(N4CCN(CC5=C(c6ccc(Cl)cc6)CCC(C)(C)C5)CC4)cc3)cc2S(=O)(=O)C(F)(F)F)CC1.[*:2]c1ccc2c(c1)CCCC2NC(=O)C1N2C(=O)C(NC(=O)C(C)NC)CCOC2CC1(C)C.[*:2]OCCCCCC(=O)[*:1]</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCCCCNC(=O)CCNCC(C(=O)N2CCN(c3ncnc4c3C(C)CC4O)CC2)c2ccc(Cl)cc2)C(C)(C)C)cc1<s><s>[*:1]N1CCC2(CC1)CCN(c1ccc(C#N)c(Cl)c1C)C2C.[*:2]N1Cc2cc3c(cc2C1)C(=O)N(C1CCC(=O)NC1=O)C3=O.[*:2]C(CCN(CC#N)c2ccc(C(=O)cc3OC)C3OC)C3CCN(C)CC2)C1=O.[*:2]CC(=O)NCCOCCOCC[*:1]</s>', 'CCn1c(-c2nonc2N)nc2c(C#CC(C)(C)O)ncc(OCCCNCCCC(=O)NCCOCCOCCOCCOCCC(=O)NC(C(=O)N3CC(O)CC3C(=O)NCc3ccc(-c4scnc4C)cc3)C(C)(C)C)c21<s><s>[*:1]OC1CCC(c2cc(OC(C)C)c(Nc3ncc4c(C(C)C)n3)c(OC)c2)CC1.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)O)O.[*:1]C(=O)CCCCCn1cc(N2CCC(OC)C[*:1])nn1</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCOCCOCCNC(=O)CCc2ccc3c(c2)[nH]c2ccncc23)C(C)(C)C)cc1<s><s>[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)NCCCCCCC(=O)CCC(=O)[*:1].[*:1]N1CCN(CCC(CSc2ccccc2))Nc2ccc(S(=O)(=O)NC(=O)C(C)(C)C)C3CCCCC3)cc2)CC1</s>', 'CCn1c(-c2nonc2N)nc2c(C#CC(C)(C)O)ncc(OCCCNCCCC(=O)NCCCCCCC(=O)NC(C(=O)N3CC(O)CC3C(=O)NCc3ccc(-c4scnc4C)cc3)C(C)(C)C)c21<s><s>[*:1]N1CCN(CCC(CSc2ccccc2)Nc2ccc(S(=O)(=O)NC(=O)c3ccc(N4CCN(CC5=C(c6ccc(Cl)cc6)CCC(C)(C)C5)CC4)cc3)cc2S(=O)(=O)C(F)(F)F)CC1.[*:2]c1ccc2c(c1)CCCC2NC(=O)C1N2C(=O)C(NC(=O)C(C)NC)CCOC2CC1(C)C.[*:2]OCCCCCC(=O)[*:1]</s>', 'COc1ccc(Cl)c(S(=O)(=O)Nc2ccc(-c3nc(OCC4CN(CCCCCCOCCCCCCOCC(=O)NC(C(=O)N5CC(O)CC5C(=O)NCc5ccc(-c6scnc6C)cc5)C(C)(C)C)CCO4)c4cn[nH]c4n3)cc2)c1<s><s>[*:1]c1cnc(OCC2NC(=O)C(F)C2CC)c2cc(OC)c(C(N)=O)cc12.[*:2]CC(=O)NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)cc1)C(C)(C)C.[*:2]C(=O)CCN(N1CCC(C#N)CC2)C(C[*:1])CC2)CC1</s>', 'CCc1cc2c(cc1N1CCC(N3CCN(C(=O)CCCCCNc4cccc5c4C(=O)N(C4CCC(=O)NC4=O)C5=O)CC3)CC1)C(C)(C)c1[nH]c3cc(C#N)ccc3c1C2=O<s><s>[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)NCCCCCCC(=O)NCCCN[*:1].[*:1]N1CCC(n2cc(OC(C)C)c(=O)c3cc(F)ccc(Oc4ccccc4S(=O)(=O)C(C)c3)nc3cnccc23)C(F)(F)C)C</s>', 'Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C(=O)CCOCCOCCC(=O)NC(C(=O)N3CC(O)CC3C(=O)NCc3ccc(-c4scnc4C)cc3)C(C)(C)C)CC2)n1<s><s>[*:1]N1CCN(CCC(CSc2ccccc2)Nc2ccc(S(=O)(=O)NC(=O)c3ccc(N4CCN(CC5=C(c6ccc(Cl)cc6)CCC(C)(C)C5)CC4)cc3)cc2S(=O)(=O)C(F)(F)F)CC1.[*:2]c1ccc2c(c1)CCCC2NC(=O)C1N(C(=O)C=O)C2=O.[*:2]OCC(=O)NCCCC(=O)[*:1]</s>', 'Cc1nn(C)c(COc2ccc(N3CCN(S(=O)(=O)N(C)C)CC3)cc2)c1-c1cccc2c(CCCOc3cccc4ccccc34)c(C(=O)NCCCCNC(=O)COc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)n(CCN3CCOCC3)c12<s><s>[*:1]CN1CCN(c2ccc(Nc3cc4c(=O)n(CC=C)n(-c5cccc(C(C)(C)O)n5)c4n3)cc2)CC1.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOC[*:1]</s>', 'COC(=O)CC1C2(C)C3=C(C)C(c4ccoc4-c4ccc(C(=O)NCCCN5CCN(c6cc(Nc7ncc(C(=O)Nc8c(C)cccc8Cl)s7)nc(C)n6)CC5)cc4)CC3OC2C2OC(=O)C3(C)C=CC(=O)C1(C)C23<s><s>[*:1]N1CCCC(n2nc(-c3ccc(Oc4ccccc4)cc3)c3c(N)ncnc32)C1.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)N(C)C(C)(C)C=CC(=O)C(C#N)C(=O)[*:1]</s>', 'O=C1CCC(N2C(=O)c3cccc(NC4CCN(CCOCCOCCNC(=O)c5ccc(-n6cc(NC(=O)c7coc(-c8ccnc(NCC(F)(F)F)c8)n7)c(C(F)F)n6)cc5)CC4)c3C2=O)C(=O)N1<s><s>[*:1]CN1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(-c5cccc(NC(=O)C=C)c5)c4n3)c(OC)c2)CC1.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)NCCCCNC(=O)CCC(=O)[*:1]</s>', 'Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(CCCCCCOCCOCCOCCCCCC(=O)Nc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)CC2)n1<s><s>[*:1]N1CCCC(n2nc(-c3ccc(Oc4ccccc4)cc3)c3c(N)ncnc32)C1.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCN(C)C(C#N)C(=O)[*:1]</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCc2cc(N)cc(CCOCCOCC#Cc3ccc(C4=NC(CC(=O)NCCS(=O)(=O)O)c5nnc(C)n5-c5sc(C)c(C)c54)cc3)c2)C(C)(C)C)cc1<s><s>[*:1]N1CCN(c2ccc(Nc3ncc4c(C)c(C(C)=O)c(=O)n(C5CCCC5)c4n3)nc2)CC1.[*:2]c1cc(-c2scnc2C)ccc1CNC(=O)C1CC(O)CN1C(=O)C(C(C)C)N1Cc2ccccc2C1=O.[*:2]OCCCCCC[*:1]</s>', 'Cc1ccc(C(=O)Nc2ccc(CN3CCN(C(=O)CCCc4cn(CCOCCOCCOCCNc5cccc6c5C(=O)N(C5CCC(=O)NC5=O)C6=O)nn4)CC3)c(C(F)(F)F)c2)cc1C#Cc1cnc2cccnn12<s><s>[*:1]N1CCCC(n2nc(-c3ccc(Oc4ccccc4)cc3)c3c(N)ncnc32)C1.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(CN(C)C(C(=O)C(C#N)C#N)C3CCCCC3)C2)C1.[*:1]</s>', 'Cc1nnc2n1-c1sc(C#Cc3cnn(-c4cccc5c4C(=O)N(C4CCC(=O)NC4=O)C5=O)c3)c(Cc3ccccc3)c1COC2<s><s>[*:1]N1CCCC(n2nc(-c3ccc(Oc4ccccc4)cc3)c3c(N)ncnc32)C1.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)NCCCCCCNC(=O)C(C)CCN(C#N)CC1)C(=O)[*:1]</s>', 'C=CC(=O)NC(CCC(=O)NCCCCCCCC(=O)NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)cc1)C(C)(C)C)C(=O)Nc1cccc(Nc2ncc(NC(=O)c3cc(NC(=O)c4cccc(C)c4)ccc3C)cn2)c1<s><s>[*:1]N1CCN(CCC(CSc2ccccc2)Nc2ccc(S(=O)(=O)NC(=O)c3ccc(N4CCN(CC5=C(c6ccc(Cl)cc6)CCC(C)(C)C5)CC4)cc3)cc2S(=O)(=O)C(F)(F)F)CC1.[*:2]c1ccc2c(c1)CCCC2NC(=O)C1N2C(=O)C(NC(=O)C(C)NC)CCOC2CC1(C)C.[*:2]OCCCCCC(=O)[*:1]</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCCCCCCN(C)CCC(CSc2ccccc2)Nc2ccc(S(=O)(=O)NC(=O)c3ccc(N4CCN(CC5=C(c6ccc(Cl)cc6)CCC(C)(C)C5)CC4)cc3)cc2S(=O)(=O)C(F)(F)F)C(C)(C)C)cc1<s><s>[*:1]c1ccc(-n2cc3cccc(C(N)=O)c3n2)cc1.[*:2]NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)cc1)C(C)(C)C.[*:2]C(=O)C(OC)CCN1CCN(c2ccc(OC)cc2c(=O)C(OC)C33OC)C(c2ccc(OC)cc33333OC)CC2=O.[nH]nc(c(=O)c2c(=O)CC1</s>', 'NC(Cc1ccc(O)c(O)c1)C(=O)NCCCCCNc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O<s><s>[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)NCCCCCCC(=O)CCC(=O)[*:1].[*:1]N1CCC(COc2ncnc3sc(C(N4CCc5cccc(C(=O)Nc5cccc5C(C(F)(F)F)c4)c3)nc2)CC1</s>', 'O=C(CCCCCOc1ccc(C=NNC(=O)c2ccc(C(=O)c3c(-c4ccc(O)cc4)sc4cc(O)ccc34)cc2)cc1)NCC#Cc1cccc2c1CN(C1CCC(=O)NC1=O)C2=O<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CCNCCCCNC(=O)CC[*:1].[*:1]N1CCN(c2ccc(OC(C)c3cc(C#Cc4OC)c(OC)c4c3o(C)nc(C)c(=O)O)c2c1</s>', 'CCn1c(-c2nonc2N)nc2c(C#CC(C)(C)O)ncc(OCCCNCCCC(=O)NCCOCCOCCOCCNc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)c21<s><s>[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCCCCCCn1cc(CCC(=O)[*:1])nn1.[*:1]N1CCN(CC#Cc2ccc(OCCCc3sc(N4c(C#Cc5cccc(C#N)c5c5C4)c3)nc2C(=O)N(OC)CC1</s>', 'O=C1CCC(N2C(=O)c3ccc(NCCCCCCNCCCONC(=O)c4ccc(F)c(F)c4Nc4ccc(I)cc4F)cc3C2=O)C(=O)N1<s><s>[*:1]CCN1CCN(c2ccc(Nc3cc(N(C)C(=O)Nc4c(Cl)c(OC)cc(OC)c4Cl)ncn3)cc2)CC1.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)NCCOCCOCCOCCOC[*:1]</s>', 'Cc1[nH]c(C=C2C(=O)Nc3ccc(F)cc32)c(C)c1C(=O)NCCCCCCCCNc1ccc2c(c1)C(=O)N(C1CCC(=O)NC1=O)C2=O<s><s>[*:1]OC1CCN(c2ccc(Nc3cc(N(C)C(=O)Nc4c(Cl)c(OC)cc(OC)c4Cl)ncn3)cc2)CC1.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)NCCOCCOC[*:1]</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCCNC(=O)c2ccc(-c3ccc(N4CCN(C)CC4)c(NC(=O)c4c[nH]c(=O)cc4C(F)(F)F)c3)cc2)C(C)(C)C)cc1<s><s>[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)NCCOCCOCCOCC[*:1].[*:1]N1CCC(c2cc(OC(C)C)c(Nc3ncc4ccccc4S(=O)(=O)C(C)C)n3)cc2C)CC1</s>', 'Nc1nc2c(c(=O)[nH]1)[n+](Cc1ccc(F)cc1)cn2C1OC(COP(=O)(O)OP(=O)(O)NCCOCCOCCOCCOCCOCCOCCOCCC(=O)Nc2cccc3c2CN(C2CCC(=O)NC2=O)C3=O)C(O)C1O<s><s>[*:1]N1CCCC(n2nc(-c3ccc(Oc4ccccc4)cc3)c3c(N)ncnc32)C1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]CC(=O)NCCOCCOCCN(C)C(C)(C)C=CC(=O)[*:1]</s>', 'CCN(CCCCCCCCCC(=O)NC(C(=O)N1CC(O)CC1C(=O)NC(C)c1ccc(-c2scnc2C)cc1)C(C)(C)C)CCOc1ccc(C(=O)c2c(-c3ccc(O)cc3)sc3cc(O)ccc23)cc1<s><s>[*:1]N1CCN(c2ccc(Nc3ncc4c(C)c(C(C)=O)c(=O)n(C5CCCC5)c4n3)nc2)CC1.[*:2]NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)cc1)C(C)(C)C.[*:2]C(=O)CCC(=O)NCCOCCOCCn1cc(C[*:1])nn1</s>', 'O=C(CCc1ccc2c(c1)[nH]c1ccncc12)NCCCCCOc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O<s><s>[*:1]c1cc2nccc(Nc3ccc4scnc4c3)c2cc1S(=O)(=O)C(C)(C)C.[*:2]Nc1cccc2c1CN(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CCCCCn1cc(C(CCCCC(=O)NC(=O)C(N)C)C(F)n2)CC1</s>', 'NC(=O)c1c(-c2ccc(Oc3ccc(F)cc3F)cc2)nn(C2CCCN(C(=O)OCCOCCOCCOCCOCCNc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)C2)c1N<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CCNCCOCCNC(=O)CC[*:1].[*:1]N1CCC(c2cc(OC(C)C)c(Nc3ncc(Cl)c(Nc4ccccc4S(=O)(=O)C(C)C)n3)cc2C)CC1</s>', 'O=C1CCC(N2C(=O)c3cccc(N4CCN(C(=O)C5CCN(c6ccc(NC(=O)c7cnc(Oc8ccccc8)nc7)cc6)CC5)CC4)c3C2=O)C(=O)N1<s><s>[*:2]c1ccc2c(c1)C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]N1CCC(CN2CCN(C(=O)NC(=O)C(c3ccc(F)cc3)C(C)(C)CC2)C.[*:2]N1CCN(c2ccc(OC)cc2)nc(Nc3ccc(C#N)c(Cl)cc3)CC2)CC1</s>', 'NC(=O)c1c(-c2ccc(Oc3ccccc3)cc2)nn2c1NCCC2C1CCN(C(=O)CCCN2CCN(C(=O)CNc3ccc4c(c3)C(=O)N(C3CCC(=O)NC3=O)C4=O)CC2)CC1<s><s>[*:1]Oc1cc2nccc(Nc3ccc(NC(=O)C4(C(=O)Nc5ccc(F)cc5)CC4)cc3F)c2cc1OC.[*:2]NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)cc1)C(C)(C)C.[*:2]C(=O)CC[*:1]</s>', 'CCN(C)S(=O)(=O)Nc1ccc(F)c(C(=O)c2c[nH]c3ncc(-c4cnc(N5CCN(C(=O)CCCOc6cccc7c6CN(C6CCC(=O)NC6=O)C7=O)CC5)nc4)cc23)c1F<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCOCCNC(=O)CC[*:1].[*:1]N1CCC(n2cc(OC(C)C)c(-c3cc(F)c(Nc4ccccc4S(=O)(=O)C(C)C)n3)cc2C)CC1</s>', 'CNC(=O)c1ccccc1Nc1cc(Nc2ccc(N3CCN(CCCCCCCCNc4cccc5c4C(=O)N(C4CCC(=O)NC4=O)C5=O)CC3)cc2OC)ncc1C(F)(F)F<s><s>[*:2]c1ccc2c(c1)C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOC[*:1].[*:1]CNC(=O)c1ccc(NC(=O)C2CCC(F)c(CN(C#N)=O)c3)c(cc4)ccc(F)c2)CC1</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)COCCOCCOCCNC(=O)CC2N=C(c3ccc(C#CCN)cc3)c3c(sc(C)c3C)-n3c(C)nnc32)C(C)(C)C)cc1<s><s>[*:2]c1ccc2c(c1)C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]N1CCC(CN2CCN(C(=O)NC(=O)C(CC)C(N)=O)CC2)cc1.[*:2]N1CCC(c3ccc(Cl)cc3)c(N)=O)c(OC)c2cc1</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)COCCOCCOCCNCCCONC(=O)c2ccc(F)c(F)c2Nc2ccc(I)cc2F)C(C)(C)C)cc1<s><s>[*:1]N1CCN(c2ccc(Nc3ncc4c(C)c(C(C)=O)c(=O)n(C5CCCC5)c4n3)nc2)CC1.[*:2]NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)cc1)C(C)(C)C.[*:2]C(=O)CCC(=O)NCCOCCOCCOCCOCCn1cc(C[*:1])nn1</s>', 'COc1cc(N=Nc2cccc3c2CN(C2CCC(=O)NC2=O)C3=O)cc(OC)c1OCC(=O)NCCCNC(=O)CC1N=C(c2ccc(Cl)cc2)c2c(sc(C)c2C)-n2c(C)nnc21<s><s>[*:1]N1CCN(C(=O)N2C(c3ccc(C#N)c(Nc4nccc5[nH]ncc34)c4ccccc34)c2)CC1.[*:2]NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)cc1)C(C)(C)C.[*:2]C(=O)CCOCCOCCOCC[*:1]</s>', 'CN(CCCCc1cn(CCOCCOCCOCCOCCOCCNc2cccc3c2C(=O)N(C2CCC(=O)NC2=O)C3=O)nn1)Cc1ccc(-c2[nH]c3cc(F)cc4c3c2CCNC4=O)cc1<s><s>[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)NCCOCCOCCOCC[*:1].[*:1]N1CCC(c2cc(OC(C)C)c(Nc3ccc(F)c(Nc4ccccc4S(=O)(=O)C(C)C)n3)cc2C)CC1</s>', 'CN(CCCCCCCCNc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O)CCC(CSc1ccccc1)Nc1ccc(S(=O)(=O)NC(=O)c2ccc(N3CCN(CC4=C(c5ccc(Cl)cc5)CCC(C)(C)C4)CC3)cc2)cc1S(=O)(=O)C(F)(F)F<s><s>[*:2]NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)cc1)C(C)(C)C.[*:2]C(=O)CCCCCCCCCC[*:1].[*:1]N1CCN(c2ccc(Nc3ncc4c(C(=O)n(C)c5cccc(NC(=O)C5)c4)cc3)CC2)CC1</s>', 'CCC(=O)NC1CCC(n2c(=O)cc(C)c3cnc(Nc4ccc(N5CCN(C(=O)CCCC#Cc6ccc7c(c6)CN(C6CCC(=O)NC6=O)C7=O)CC5)cc4OC)nc32)CC1<s><s>[*:1]N1CCN(c2ccc(Nc3ncc4c(C)c(C(C)=O)c(=O)n(C5CCCC5)c4n3)nc2)CC1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CCOCCOCCOCCn1cc(CCC(=O)[*:1])nn1</s>', 'Cc1[nH]c(C=C2C(=O)Nc3ccc(S(=O)(=O)Cc4c(Cl)cccc4Cl)cc32)c(C)c1C(=O)NCCOCCOCCNc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC[*:1].[*:1]N1CCN(c2ccc(OC(C)C)c(Nc3ncc4cc(Cl)c(Nc5ccccc5)c(OC)n4S(=O)(=O)C)n3)cc2C)CC1</s>', 'CCC(=O)NC1CCC(n2c(=O)cc(C)c3cnc(Nc4ccc(N5CCN(C(=O)CCCCCCCC#Cc6ccc7c(c6)C(=O)N(C6CCC(=O)NC6=O)C7)CC5)cc4OC)nc32)CC1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CCNCCOCCNC(=O)CC[*:1].[*:1]N1CCC(c2cc(OC(C)C)c(Nc3ncc(Cl)c(Nc4ccccc4S(=O)(=O)C(C)C)n3)cc2C)CC1</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)COCCOCCOc2ccc(C=NNC(=O)c3cccc(Oc4c(-c5ccc(O)cc5)sc5cc(O)ccc45)c3)cc2)C(C)(C)C)cc1<s><s>[*:1]N1CCC(c2cc(OC(C)C)c(Nc3cc(Nc4ccccc4C(=O)NC)c(C(F)(F)F)cn3)cc2C)CC1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]CCOCCOCCOCCn1cc(CCC(=O)[*:1])nn1</s>', 'COc1cc(N=Nc2cccc3c2CN(C2CCC(=O)NC2=O)C3=O)cc(OC)c1OCC(=O)NCCCCNC(=O)CC1N=C(c2ccc(Cl)cc2)c2c(sc(C)c2C)-n2c(C)nnc21<s><s>[*:1]OC(=O)C(C)C=CC1(C)OC(=O)C23CC=C(C(=O)OC)CCC2(OC(C)=O)C1CC3.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC[*:1]</s>', 'CCC(C)C(C(CC(=O)N1CCCC1C(OC)C(C)C(=O)NC(C)C(O)c1ccccc1)OC)N(C)C(=O)C(NC(=O)C(C(C)C)N(C)CCOCCOCCOCCNc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O)C(C)C<s><s>[*:1]N1CCN(c2ccc(Nc3ncc4c(C)c(C(C)=O)c(=O)n(C5CCCC5)c4n3)nc2)CC1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CCCCCOCCCCC[*:1]</s>', 'O=C1CCC(N2C(=O)c3ccc(OCCOCCOCCOCCN4CCN(Cc5ccc6nc(NC(=O)c7cccc(C(F)(F)F)c7)n(C7CCC(CO)CC7)c6c5)CC4)cc3C2=O)C(=O)N1<s><s>[*:1]N1CCN(c2ccc(Nc3cc4c(C)c(C(C)=O)c(=O)n(C5CCCC5)c4n3)nc2)CC1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]CCOCCOCCOCC[*:1]</s>', 'Cc1cc(-c2ccc(C(=O)NCCCCCCNC(=O)COc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)cc2N2CCC(c3[nH]cnc3C)CC2)ccc1F<s><s>[*:1]N1CCN(c2ccc(Nc3ncc4c(C)c(C(C)=O)c(=O)n(C5CCCC5)c4n3)nc2)CC1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]CCOCCOCCOCC(=O)[*:1]</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)COCCOCCOCCOCCOCCOCCOc2ccc(C=NNC(=O)c3ccc(C(=O)c4c(-c5ccc(O)cc5)sc5cc(O)ccc45)cc3)cc2)C(C)(C)C)cc1<s><s>[*:1]N1CCN(c2ccc(Nc3ncc4c(C)c(C(C)=O)c(=O)n(C5CCCC5)c4n3)nc2)CC1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]CCCCC(=O)[*:1]</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)COCCOCCOCCOCCOc2ccc(C=NNC(=O)c3c(-c4ccc(O)cc4)sc4cc(O)ccc34)cc2)C(C)(C)C)cc1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CCCCC[*:1].[*:1]N1CCN(c2ccc(Nc3ncc(C#N)c(Nc4ccccc5)c4)cc3)CC2)CC1</s>', 'CN1CCN(c2ccc(-c3ccc(C(=O)NCCOCCOCCOCCOCCOCCNc4cccc5c4C(=O)N(C4CCC(=O)NC4=O)C5=O)cc3)cc2NC(=O)c2c[nH]c(=O)cc2C(F)(F)F)CC1<s><s>[*:1]N1CCC2(CC1)CC(C)N(c1ccc(C#N)c(Cl)c1)C2.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CCCCC[*:1]</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCC(=O)N2CCN(Cc3cccc(-c4ccc(N5CCN(C)CC5)c(NC(=O)c5c[nH]c(=O)cc5C(F)(F)F)c4)c3)CC2)C(C)(C)C)cc1<s><s>[*:1]N1CCN(Cc2ccc(NC(=O)c3ccc(C)c(C#Cc4ccccc4[nH]ncc5)c4)cc2)CC1.[*:2]c1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]OCC(=O)NCCCCCC[*:1]</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCCCCOCCCCOCCCCOc2cc3ncnc(Nc4ccc(F)c(Cl)c4)c3cc2NC(=O)C=CCN(C)C)C(C)(C)C)cc1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC[*:1].[*:1]c1ccc(N2CCN(C(=O)CC3ccc(C)c(Nc4nccc(-c5ccccc5)n4)c(OC)c3)cc2)CC1</s>', 'CCCS(=O)(=O)Nc1ccc(F)c(C(=O)c2c[nH]c3ncc(-c4ccc(CNC(=O)CCCCC(=O)NC(C(=O)N5CC(O)CC5C(=O)NCc5ccc(-c6scnc6C)cc5)C(C)(C)C)cc4)cc23)c1F<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CCCCC[*:1].[*:1]NC(=O)CC1.[*:1]C(=O)N1CCC(Oc3ccc(C#N)c(Cl)c3)C2)CC1</s>', 'CCCNNC(=O)c1ccc(-c2ccc(NC(=O)CCCCC(=O)NC(C(=O)N3CC(O)CC3C(=O)NC(C)c3ccc(-c4scnc4C)cc3)C(C)(C)C)cc2)cc1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CCCCC[*:1].[*:1]NC(=O)CCCCC1.[*:1]N1CCC(c2cc(Nc3ncc5ccccc5)c(F)c(OC)c3)cc2)CC1</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCCCCCCCCCNC(=O)CCNCC(C(=O)N2CCN(c3ncnc4c3C(C)CC4O)CC2)c2ccc(Cl)cc2)C(C)(C)C)cc1<s><s>[*:1]N1CCN(c2ccc(Nc3ncc4c(C)c(C(C)=O)c(=O)n(C5CCCC5)c4n3)nc2)CC1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]CCCCC(=O)[*:1]</s>', 'CCn1c(=O)n(CC(=O)NCCOCCNC(=O)COc2cccc3c2C(=O)N(C2CCC(=O)NC2=O)C3=O)c(=O)c2cc(C(N)=O)c(N)nc21<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CCCCC[*:1].[*:1]NC(=O)CC1(c2ccc(C)cc3)c(Nc4ccccc4)c(OC)c3)cc2C1</s>', 'CCC(=C(c1ccc(O)cc1)c1ccc(OCCN(C)C(=O)COCCOCCOCCNC(=O)C(NC(=O)C2CCCN2C(=O)C(NC(=O)C(C)NC)C2CCCCC2)C(c2ccccc2)c2ccccc2)cc1)c1ccccc1<s><s>[*:1]N1CCC2(CC1)CC(C)N(c1ccc(C#N)c(Cl)c1)C2.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CCCCC(=O)NC[*:1]</s>', 'COc1cc(Cl)c(-c2cc(Cl)ccc2Cl)cc1C(=O)N1CCN(C(=O)CCC(=O)NCCCCNc2cccc3c2C(=O)N(C2CCC(=O)NC2=O)C3=O)CC1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CCCCC[*:1].[*:1]NC1CCN(c2ccc(C(=O)Nc3ccc(C)C)c(Nc4cc(Cl)c(Nc5ccccc5)c4S(=O)(=O)C(C)n3)cc2)CC1</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCCCCCCCCCC(=O)N2CCN(C(=O)c3cc(Cc4n[nH]c(=O)c5ccccc45)ccc3F)CC2)C(C)(C)C)cc1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC[*:1].[*:1]N1CCN(c2ccc(Nc3cc(C)c(Nc4nccc(-c5ccccc5)n4)c(OC)c3)CC2)CC1</s>', 'CCOc1cc2nc(CCC(C)(C)C(=O)NCCOCCNc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)n(C)c2cc1NC(=O)c1cccc(C(F)(F)F)n1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC[*:1].[*:1]CN1CCN(c2ccc(Nc3ccc(C(=O)Nc4cc(C)c(OC)c(Nc5ccccc5)cc4S(=O)(=O)C(C)n3)cc2)CC1</s>', 'O=C1CCC(N2C(=O)c3cccc(NCCOCCOCCOCCNCCCONC(=O)c4ccc(F)c(F)c4Nc4ccc(I)cc4F)c3C2=O)C(=O)N1<s><s>[*:1]NC1CCN(c2ccc(Nc3ncc4c(C)cc(=O)n(C5CCC(NC(=O)CC)CC5)c4n3)nc2)CC1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]CC(=O)NC(=O)C[*:1]</s>', 'Nc1c(-c2nc3cc(N4CCN(C(=O)CCCCCCNc5cccc6c5C(=O)N(C5CCC(=O)NC5=O)C6=O)CC4)ccc3[nH]2)c(=O)[nH]c2cccc(F)c12<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC[*:1].[*:1]CN1CCN(c2ccc(C(=O)C)Nc3ccc(Cl)c(Nc4ccccc4S(=O)(=O)C(C)C)c3)cc2)CC1</s>', 'COc1cc(OC)c(C=CS(=O)(=O)Cc2ccc(OC)c(CC(=O)NCC(=O)Nc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)c2)c(OC)c1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CCCCC[*:1].[*:1]CN1CCN(c2ccc(OC(=O)C(=O)NC(=O)c3ccc(F)cc3)c(Nc4ccccc4)cc3)CC1</s>', 'COc1cc(-c2cn(C)c(=O)c3cnccc23)cc(OC)c1CN1CCN(CCOCCOCCOc2cc(-c3scnc3C)ccc2CNC(=O)C2CC(O)CN2C(=O)C(NC(C)=O)C(C)(C)C)CC1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC[*:1].[*:1]CN1CCN(c2ccc(Nc3ccc(C(=O)Nc4cc(C)c(OC)c(Nc5ccccc5)c4S(=O)(=O)C(C)n3)cc2)CC1</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCCCCOc2ccc(C=NNC(=O)c3cccc(Oc4c(-c5ccc(O)cc5)sc5cc(O)ccc45)c3)cc2)C(C)(C)C)cc1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CCCCC[*:1].[*:1]CN1CCN(c2ccc(NC(=O)c3ccc(C)Nc4cc(C)c(Cl)c4ccccc4)cc3)CC1</s>', 'CCOC(=O)c1[nH]c2cc(Cl)ccc2c1C(C(=O)NC(C)(C)C)N(Cc1cc(F)c(F)c(F)c1)C(=O)c1cccc(OCCC#Cc2cccc3c2CN(C2CCC(=O)NC2=O)C3=O)c1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC[*:1].[*:1]CN1CCN(c2ccc(Nc3ccc(C(=O)Nc4cc(C)c(OC)c(Nc5ccccc5)c(OC)c4S(=O)(=O)C)C)n3)cc2)CC1</s>', 'CCCS(=O)(=O)Nc1ccc(F)c(-n2cc(-c3cncnc3)c3nc(N(C)C4CCN(CC5CN(C(=O)CCOCCOCCOCCNc6cccc7c6C(=O)N(C6CCC(=O)NC6=O)C7=O)C5)CC4)ccc32)c1F<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC[*:1].[*:1]CN1CCN(c2ccc(Nc3ccc(C(=O)Nc4cc(C)c(OC)c(Nc5ccccc5)c4S(=O)(=O)C(C)n3)cc2)CC1</s>', 'COC(=O)C1=CCC23CCC(C(C)(C=CC=C(C)C(=O)NC4CCN(c5cccc6c5C(=O)N(C5CCC(=O)NC5=O)C6=O)CC4)OC2=O)C3(OC(C)=O)CC1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC[*:1].[*:1]CN1CCN(c2ccc(Nc3ccc(C(=O)Nc4cc(C)c(OC)c(Nc5ccccc5)c4S(=O)(=O)C(C)n3)cc2)CC1</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCOCCOCCOCCNC(=O)CCc2ccc3c(c2)[nH]c2ccncc23)C(C)(C)C)cc1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC[*:1].[*:1]CN1CCN(c2ccc(Nc3ccc(C(=O)Nc4cc(C)c(OC)c(Nc5ccccc5)c4S(=O)(=O)C(C)n3)cc2)CC1</s>', 'CNC(C)C(=O)NC(C(=O)N1CCCC1c1nc(C(=O)c2cccc(OCCOCCOCCOCCOCC(=O)N3CCN(c4cc(Nc5ncc(C(=O)Nc6c(C)cccc6Cl)s5)nc(C)n4)CC3)c2)cs1)C1CCCCC1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC[*:1].[*:1]CN1CCN(c2ccc(Nc3ccc(C(=O)Nc4cc(C)c(OC)c(Nc5ccccc5)c4S(=O)(=O)C(C)n3)cc2)CC1</s>', 'CN(c1ncccc1CNc1nc(Nc2ccc(C(=O)NCc3cn(CCOCCOCCOCCNc4cccc5c4C(=O)N(C4CCC(=O)NC4=O)C5=O)nn3)cc2)ncc1C(F)(F)F)S(C)(=O)=O<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC[*:1].[*:1]CN1CCN(c2ccc(Nc3ccc(C(=O)Nc4cc(C)c(OC)c(Nc5ccccc5)c4S(=O)(=O)C(C)n3)cc2)CC1</s>', 'O=C1CCC(N2C(=O)c3cccc(NCCOCCOCCn4cc(CCCC(=O)NC5CCN(c6ncc(C(=O)Nc7ccc(OC(F)(F)Cl)cc7)cc6-c6cc[nH]n6)C5)nn4)c3C2=O)C(=O)N1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC[*:1].[*:1]CN1CCN(c2ccc(Nc3ccc(C(=O)Nc4cc(C)c(OC)c(Nc5ccccc5)c4S(=O)(=O)C(C)n3)cc2)CC1</s>', 'CC1(C)C(NC(=O)c2ccc(N3CCN(CC4CCN(c5ccc6c(c5)C(=O)N(C5CCC(=O)NC5=O)C6=O)CC4)CC3)cc2)C(C)(C)C1Oc1cnc(C#N)c(C(F)(F)F)c1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC[*:1].[*:1]CN1CCN(c2ccc(Nc3ccc(C(=O)Nc4cc(C)c(OC)c(Nc5ccccc5)c4S(=O)(=O)C(C)n3)cc2)CC1</s>', 'NC(=O)c1c(-c2ccc(Oc3ccccc3)cc2)nn2c1NCCC2C1CCN(C(=O)CCCc2ccc(NC(=O)CNc3ccc4c(c3)C(=O)N(C3CCC(=O)NC3=O)C4=O)cc2)CC1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC[*:1].[*:1]CN1CCN(c2ccc(Nc3ccc(C)c(Nc4cc(OC)c(Nc5ccccc5)nH])c4S(=O)(=O)C(C)n3)cc2)CC1</s>', 'CCC(=O)N1CCN(c2nc(NCCC(=O)N(C)CCOCCOCCOCCOCCOc3ccc4c(c3)CN(C3CCC(=O)NC3=O)C4=O)nc3c(F)c(-c4cc(O)cc5ccccc45)c(Cl)cc23)CC1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC[*:1].[*:1]CN1CCN(c2ccc(Nc3ccc(C(=O)Nc4cc(C)c(OC)c(Nc5ccccc5)c(OC)c4S(=O)(=O)C)C)n3)cc2)CC1</s>', 'Cc1ccc(NC(=O)c2ccc(CN3CCN(C(=O)CCCc4cn(CCOCCNc5cccc6c5C(=O)N(C5CCC(=O)NC5=O)C6=O)nn4)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC[*:1].[*:1]CN1CCN(c2ccc(Nc3ccc(C(=O)Nc4cc(C)c(OC)c(Nc5ccccc5)c(OC)c4S(=O)(=O)C)C)n3)cc2)CC1</s>', 'Cc1sc2c(c1C)C(c1ccc(Cl)cc1)=NC(CC(=O)NC(C)c1ccc(OCCOCCNc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)cc1)c1nnc(C)n1-2<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC[*:1].[*:1]CN1CCN(c2ccc(Nc3ccc(C(=O)Nc4cc(C)c(OC)c(Nc5ccccc5)c(OC)c4S(=O)(=O)C)C2)CC1</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)COCCOCCOCCOCCOCC(=O)N2CC(Nc3cc(C(=O)NCC(O)CN4CCc5ccccc5C4)ncn3)C2)C(C)(C)C)cc1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC[*:1].[*:1]CN1CCN(c2ccc(Nc3ccc(C(=O)Nc4cc(C)c(OC)c(Nc5ccccc5)c4S(=O)(=O)C(C)n3)cc2)CC1</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCOCCNC(=O)c2ccc(-c3ccc(N4CCN(C)CC4)c(NC(=O)c4c[nH]c(=O)cc4C(F)(F)F)c3)cc2)C(C)(C)C)cc1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC[*:1].[*:1]CN1CCN(c2ccc(Nc3ccc(C(=O)Nc4cc(C)c(OC)c(Nc5ccccc5)c(OC)c4S(=O)(=O)C)C)n3)cc2)CC1</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCCCCN(C)CCC(CSc2ccccc2)Nc2ccc(S(=O)(=O)NC(=O)c3ccc(N4CCN(CC5=C(c6ccc(Cl)cc6)CCC(C)(C)C5)CC4)cc3)cc2S(=O)(=O)C(F)(F)F)C(C)(C)C)cc1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC[*:1].[*:1]CN1CCN(c2ccc(Nc3ccc(C(=O)Nc4cc(C)c(OC)c(Nc5ccccc5)c4S(=O)(=O)C(C)n3)cc2)CC1</s>', 'CCOc1cc2nc(CCC(C)(C)C(=O)NCCCCNc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)n(C)c2cc1NC(=O)c1cccc(C(F)(F)F)n1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC[*:1].[*:1]CN1CCN(c2ccc(Nc3ccc(C(=O)Nc4cc(C)c(OC)c(Nc5ccccc5)c4S(=O)(=O)C(C)n3)cc2)CC1</s>', 'CC1CC(O)c2ncnc(N3CCN(C(=O)C(CNCCC(=O)NCCCNc4cccc5c4C(=O)N(C4CCC(=O)NC4=O)C5=O)c4ccc(Cl)cc4)CC3)c21<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC[*:1].[*:1]CN1CCN(c2ccc(Nc3ccc(C(=O)Nc4cc(C)c(OC)c(Nc5ccccc5)c4S(=O)(=O)C(C)n3)cc2)CC1</s>', 'Cc1cc(C(C(=O)N2CC(O)CC2C(=O)NC(CC(=O)N2CCC(N3CCC(C#Cc4ccc(C(=O)NC5C(C)(C)C(Oc6ccc(C#N)c(Cl)c6)C5(C)C)cc4)CC3)CC2)c2ccc(C#N)cc2)C(C)C)on1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC[*:1].[*:1]CN1CCN(c2ccc(Nc3ccc(C(=O)Nc4cc(C)c(OC)c(Nc5ccccc5)c4S(=O)(=O)C(C)n3)cc2)CC1</s>', 'O=C1CCC(N2C(=O)c3cccc(NCCCCCC4CCN(CCNC(=O)c5cc6c(o5)C(=O)c5ccccc5C6=O)CC4)c3C2=O)C(=O)N1<s><s>[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)CNCCCCC[*:1].[*:1]CN1CCN(c2ccc(Nc3ccc(C(=O)Nc4cc(C)c(OC)c(Nc5ccccc5)c4S(=O)(=O)C(C)n3)cc2)CC1</s>', 'CCOc1cc(C(C)(C)C)ccc1C1=NC(C)(c2ccc(Cl)cc2)C(C)(c2ccc(Cl)cc2)N1C(=O)N1CCN(CC(=O)NCCOCCOCCOCCn2cc(C(=O)N3CCCC(n4nc(-c5ccc(Oc6ccccc6)cc5)c5c(N)ncnc54)C3)nn2)CC1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1[nH]c(C=C2C(=O)Nc3ccc(F)cc32)c(C)c1C(=O)NCCCCCCCCNC(=O)COc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Nc1c(-c2nc3cc(N4CCN(C(=O)CCCCNc5cccc6c5C(=O)N(C5CCC(=O)NC5=O)C6=O)CC4)ccc3[nH]2)c(=O)[nH]c2cccc(F)c12<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCCCOc2ccc(C=NNC(=O)c3cccc(C(=O)c4c(-c5ccc(O)cc5)sc5cc(O)ccc45)c3)cc2)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'COc1cc(Nc2ncc3c(n2)-c2ccc(Cl)cc2C(c2c(F)cccc2OC)=NC3)ccc1C(=O)NCCCOCCOCCOCCCNC(=O)COc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CC(C)c1c(C(=O)Nc2ccccc2)c(-c2cccc(OCCCc3cn(CCCOCC#Cc4cccc5c4CN(C4CCC(=O)NC4=O)C5=O)nn3)c2)c(-c2ccc(F)cc2)n1CCC(O)CC(O)CC(=O)O<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CCCS(=O)(=O)Nc1ccc(F)c(-n2cc(-c3cncnc3)c3nc(N(C)C4CCN(C(=O)CCOCCOCCOCCOCCC(=O)NC(C(=O)N5CC(O)CC5C(=O)NCc5ccc(-c6scnc6C)cc5)C(C)(C)C)CC4)ccc32)c1F<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ccc(NC(=O)c2cccc(C(F)(F)F)c2)cc1N1Cc2cnc(Nc3ccc(N4CCN(CCOc5cccc6c5C(=O)N(C5CCC(=O)NC5=O)C6=O)CC4)nc3)nc2N(C)C1=O<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CNC(=O)c1ccccc1Nc1cc(Nc2ccc(N3CCN(CCCCCCNC(=O)COc4cccc5c4C(=O)N(C4CCC(=O)NC4=O)C5=O)CC3)cc2OC)ncc1C(F)(F)F<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Nc1nc(Nc2ccc(CNC(=O)CCCCCNc3cccc4c3CN(C3CCC(=O)NC3=O)C4=O)cc2)nn1-c1ccc(-c2ccccc2)nn1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCCCCCCC(=O)N2CCC(NC3=Nc4cc(F)ccc4N(CC(F)F)c4ccc(Cl)cc43)C2)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'COc1cc(N2CCN(C)CC2)c(NC(=O)C(C#N)=CC(C)(C)NC(=O)CCCCCC(=O)NC(C(=O)N2CC(O)CC2C(=O)NCc2ccc(-c3scnc3C)cc2)C(C)(C)C)cc1Nc1nccc(-c2cn(C)c3ccccc23)n1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCCCCCCCCCNCCCONC(=O)c2ccc(F)c(F)c2Nc2ccc(I)cc2F)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1nn(C)c(COc2ccc(N3CCN(S(=O)(=O)N(C)C)CC3)cc2)c1-c1cccc2c(CCCOc3cccc4ccccc34)c(C(=O)O)n(CCN3CCN(C(=O)CCC(=O)NCCCOCCOCCOCCCNC(=O)COc4cccc5c4C(=O)N(C4CCC(=O)NC4=O)C5=O)CC3)c12<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CCN(C)S(=O)(=O)Nc1ccc(F)c(C(=O)c2c[nH]c3ncc(-c4ccc(N5CCN(C(=O)CCCOc6cccc7c6CN(C6CCC(=O)NC6=O)C7=O)CC5)cc4)cc23)c1F<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CNC(C)C(=O)NC(C(=O)N1CCCC1c1nc(C(=O)c2cccc(OCCOCCOCCOCCNC(=O)CCCCCCOc3c(OC)ccc4c(Nc5c(Cl)cncc5Cl)cc(=O)oc34)c2)cs1)C1CCCCC1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CN(CCC(CSc1ccccc1)Nc1ccc(S(=O)(=O)NC(=O)c2ccc(N3CCN(CC4=C(c5ccc(Cl)cc5)CCC(C)(C)C4)CC3)cc2)cc1S(=O)(=O)C(F)(F)F)C(=O)CCCn1cc(COCCOCCOCCNc2cccc3c2C(=O)N(C2CCC(=O)NC2=O)C3=O)nn1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)c2ccc(-c3ccc(N4CCN(C)CC4)c(NC(=O)c4c[nH]c(=O)cc4C(F)(F)F)c3)cc2)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)COCCOc2ccc(C=NNC(=O)c3ccc(-c4c(-c5ccc(O)cc5)sc5cc(O)ccc45)cc3)cc2)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CCOC(=O)c1[nH]c2cc(Cl)ccc2c1C(C(=O)NC(C)(C)C)N(Cc1ccc(Cl)cc1)C(=O)CCCCC#Cc1cccc2c1CN(C1CCC(=O)NC1=O)C2=O<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CN(CCC(=O)N1CCC(C2CCNc3c(C(N)=O)c(-c4ccc(Oc5ccccc5)cc4)nn32)CC1)CC1CCN(c2ccc3c(c2)C(=O)N(C2CCC(=O)NC2=O)C3=O)CC1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CCC(C)C(C(CC(=O)N1CCCC1C(OC)C(C)C(=O)NC(C)C(O)c1ccccc1)OC)N(C)C(=O)C(NC(=O)C(C(C)C)N(C)CCOCCOCCNc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O)C(C)C<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CN(c1ncccc1CNc1nc(Nc2ccc(NC(=O)c3cn(CCCCCCCCNc4cccc5c4C(=O)N(C4CCC(=O)NC4=O)C5=O)nn3)cc2)ncc1C(F)(F)F)S(C)(=O)=O<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'O=C1CCC(N2Cc3c(NC(=O)CCCCN4CCN(c5ccc(Nc6cc7c(N8CCC(CO)CC8)nc(Nc8ccc(F)cc8)nc7cn6)nc5)CC4)cccc3C2=O)C(=O)N1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1cc(C(C(=O)N2CC(O)CC2C(=O)NC(CC(=O)N2CCC(N3CCC(C#Cc4ccc(S(=O)(=O)CC(C)(O)C(=O)Nc5ccc(C#N)c(C(F)(F)F)c5)cc4)CC3)CC2)c2ccc(-c3scnc3C)cc2)C(C)C)on1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'O=C(CCC(=O)N1CCN(c2nc(N3CCOCC3)nc(-n3c(C(F)F)nc4ccccc43)n2)CC1)NCCCC(=O)Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)COCCOCCOCCOc2cc(F)c(C3c4[nH]c5ccccc5c4CC(C)N3CC(C)(C)F)c(F)c2)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1[nH]c(C=C2C(=O)Nc3ccc(F)cc32)c(C)c1C(=O)NCCCCCCNc1ccc2c(c1)C(=O)N(C1CCC(=O)NC1=O)C2=O<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'O=C1CCC(N2C(=O)c3cccc(NCCOCCOCCOCCOCCOCC(=O)N4CCN(C(=O)c5cc(Cc6n[nH]c(=O)c7ccccc67)ccc5F)CC4)c3C2=O)C(=O)N1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CN1CCN(c2ccc(-c3cccc(CN4CCN(CCNC(=O)CCCCNc5cccc6c5C(=O)N(C5CCC(=O)NC5=O)C6=O)CC4)c3)cc2NC(=O)c2c[nH]c(=O)cc2C(F)(F)F)CC1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CCOc1cc(C(C)(C)C)ccc1C1=NC(c2ccc(Cl)cc2)C(c2ccc(Cl)cc2)N1C(=O)N1CCN(CCC#Cc2cccc3c2CN(C2CCC(=O)NC2=O)C3=O)CC1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CCC(=C(c1ccc(O)cc1)c1ccc(OCCN(C)C(=O)COCCNC(=O)C(NC(=O)C2CCCN2C(=O)C(NC(=O)C(C)NC)C2CCCCC2)C(c2ccccc2)c2ccccc2)cc1)c1ccccc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'C=CC(=O)NC(CCC(=O)NCCCCCCCCNc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O)C(=O)Nc1cccc(Nc2ncc(NC(=O)c3cc(NC(=O)c4cccc(C(F)(F)F)c4)ccc3C)cn2)c1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCCCCCCCCCNC(=O)CCNCC(C(=O)N2CCN(c3ncnc4c3C(C)CC4O)CC2)c2ccc(Cl)cc2)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'COc1cc(-c2cn(C)c(=O)c3cnccc23)cc(OC)c1CN1CCN(CCOCCOCCOCCOCCOc2cc(-c3scnc3C)ccc2CNC(=O)C2CC(O)CN2C(=O)C(NC(C)=O)C(C)(C)C)CC1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)COc2ccc(C=NNC(=O)c3cccc(C(=O)c4c(-c5ccc(O)cc5)sc5cc(O)ccc45)c3)cc2)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)COCCCOCCCCOc2ccc(N3C(=S)N(c4ccc(C#N)c(C(F)(F)F)c4)C(=O)C3(C)C)cc2)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'COc1cc(OC)c(Cl)c(NC(=O)N(C)c2cc(Nc3ccc(N4CCN(CC(=O)NCCCCCOc5cccc6c5C(=O)N(C5CCC(=O)NC5=O)C6=O)CC4)cc3)ncn2)c1Cl<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CCn1c(-c2nonc2N)nc2c(C#CC(C)(C)O)ncc(OCCCNCCCC(=O)NCCCCCCCCCCC(=O)NC(C(=O)N3CC(O)CC3C(=O)NCc3ccc(-c4scnc4C)cc3)C(C)(C)C)c21<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCCCOc2ccc(-c3csc(N4CCOC(C)(C)C4)n3)cc2)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'COc1cc(N2CCC(OCC(=O)NC(C(=O)N3CC(O)CC3C(=O)NC(C)c3ccc(-c4scnc4C)cc3)C(C)(C)C)CC2C)ccc1NC(=O)c1cccc(-c2ccn[nH]2)n1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CCN(c1cc(-c2ccc(CN3CCN(CCCCCOc4cccc5c4C(=O)N(C4CCC(=O)NC4=O)C5=O)CC3)cc2)cc(C(=O)NCc2c(C)cc(C)[nH]c2=O)c1C)C1CCOCC1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)COCCOCCOCCNC(=O)c2cc3c(cc2CS(C)(=O)=O)-c2cn(C)c(=O)c4[nH]cc(c24)CN3c2ncc(F)cc2F)C(C)(C)C)c(OC2CCNCC2)c1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)COc2ccc(C=NNC(=O)c3ccc(Oc4c(-c5ccc(O)cc5)sc5cc(O)ccc45)cc3)cc2)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'O=C(CCCCCCCCCCC(=O)N1CCN(C(=O)c2cc(Cc3n[nH]c(=O)c4ccccc34)ccc2F)CC1)NCCCCCCNc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1c(NC(=O)c2ccc(C(C)(C)C)cc2)cccc1-c1cn(C)c(=O)c(Nc2ccc(C(=O)N3CCN(CCOCCOCCNC(=O)CNc4cccc5c4C(=O)N(C4CCC(=O)NC4=O)C5=O)CC3)cc2)n1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)COc2ccc(C=NNC(=O)c3cccc(-c4c(-c5ccc(O)cc5)sc5cc(O)ccc45)c3)cc2)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1sc2c(c1C)C(c1ccc(Cl)cc1)=NC(CC(=O)Nc1ccc(N=Nc3ccc(NC(=O)COc4cccc5c4C(=O)N(C4CCC(=O)NC4=O)C5=O)cc3)cc1)c1nnc(C)n1-2<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CCN(c1cc(-c2ccc(C(=O)NCCOCCNc3cccc4c3C(=O)N(C3CCC(=O)OC3=O)C4=O)cc2)cc(C(=O)NCc2c(C)cc(C)[nH]c2=O)c1C)C1CCOCC1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1c(COc2cc(OCc3cccc(C#N)c3)c(CN3CCCCC3C(=O)NCCNC(=O)COCC(=O)Nc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)cc2Cl)cccc1-c1ccc2c(c1)OCCO2<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCCCCCCCCCNCCCCONC(=O)c2ccc(F)c(F)c2Nc2ccc(I)cc2F)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'O=C(CCCCCCC(=O)N1CCN(C(=O)c2cc(Cc3n[nH]c(=O)c4ccccc34)ccc2F)CC1)NCCNc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CCn1c(=O)n(CC(=O)NCCNC(=O)COc2cccc3c2C(=O)N(C2CCC(=O)NC2=O)C3=O)c(=O)c2cc(C(N)=O)c(N)nc21<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CCN(CCCCCCCC(=O)NC(C(=O)N1CC(O)CC1C(=O)NC(C)c1ccc(-c2scnc2C)cc1)C(C)(C)C)CCOc1ccc(Cn2c(-c3ccc(O)cc3)c(C)c3cc(O)ccc32)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CCn1c(-c2nonc2N)nc2c(C#CC(C)(C)O)ncc(OCCCNCCCC(=O)NCCOCCOCCOCC(=O)NC(C(=O)N3CC(O)CC3C(=O)NCc3ccc(-c4scnc4C)cc3)C(C)(C)C)c21<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'COc1cc(OCc2cccc(-c3ccccc3)c2C)cc(OC)c1CNCCCCNc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CC1CC(O)c2ncnc(N3CCN(C(=O)C(CNCCC(=O)NCCCCNc4cccc5c4C(=O)N(C4CCC(=O)NC4=O)C5=O)c4ccc(Cl)cc4)CC3)c21<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CC1CCC2(CCN(c3cnc(Sc4cccc(NC(=O)CCC(=O)NCCCCCCCCNc5cccc6c5C(=O)N(C5CCC(=O)NC5=O)C6=O)c4Cl)c(N)n3)CC2)C1N<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CCC(=O)NC1CCC(n2c(=O)cc(C)c3cnc(Nc4ccc(N5CCN(C(=O)CCCCCCCC#Cc6cccc7c6C(=O)N(C6CCC(=O)NC6=O)C7)CC5)cc4OC)nc32)CC1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'O=C(COCCOCCOCCOCCOCCOCCOc1ccc(C=NNC(=O)c2ccc(C(=O)c3c(-c4ccc(O)cc4)sc4cc(O)ccc34)cc2)cc1)NCC#Cc1cccc2c1CN(C1CCC(=O)NC1=O)C2=O<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'O=C1CCC(N2C(=O)c3ccc(NCCOCCOCCNCCCONC(=O)c4ccc(F)c(F)c4Nc4ccc(I)cc4F)cc3C2=O)C(=O)N1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1sc2c(c1C)C(c1ccc(Cl)cc1)=NC(CC(=O)NCCCCCCCCN1CCN(c3ccc(NC4CCC(=O)NC4=O)cc3)CC1)c1nnc(C)n1-2<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'O=C1CCC(N2C(=O)c3cccc(NCCOCCOCCOCCn4cc(CCCC(=O)NC5CCN(c6ncc(C(=O)Nc7ccc(OC(F)(F)Cl)cc7)cc6-c6cc[nH]n6)C5)nn4)c3C2=O)C(=O)N1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)COCCOCCNC(=O)CCNCC(C(=O)N2CCN(c3ncnc4c3C(C)CC4O)CC2)c2ccc(Cl)cc2)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CNC(=O)N1CCc2c(c(N3CCCc4cc(-c5cnn(C)c5)c(C(F)F)cc43)nn2C2CCN(C(=O)CCOCCOCCOCCOCCNc3ccc4c(c3)C(=O)N(C3CCC(=O)NC3=O)C4=O)CC2)C1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1[nH]c(C=C2C(=O)Nc3ccc(S(=O)(=O)Cc4c(Cl)cccc4Cl)cc32)c(C)c1C(=O)NCCCCCCNc1ccc2c(c1)C(=O)N(C1CCC(=O)NC1=O)C2=O<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1sc2c(c1C)C(c1ccc(Cl)cc1)=NC(CC(=O)NCCNC(=O)COc1ccc(N=Nc3cccc4c3CN(C3CCC(=O)NC3=O)C4=O)cc1)c1nnc(C)n1-2<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1cc(-c2ccc(C(=O)NCCOCCOCCOCCOCCNc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)cc2N2CCC(c3[nH]cnc3C)CC2)ccc1F<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCCCCCCOc2ccc(C=NNC(=O)c3ccc(-c4c(-c5ccc(O)cc5)sc5cc(O)ccc45)cc3)cc2)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CC1CCC2(C(=O)NCCOCCOCCOCCNc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)CCC3(C)C(=CCC4C5(C)CCC(O)C(C)(C)C5CCC43C)C2C1C<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CC(C=CC1=C(C)C(=NOCC(=O)NCCOCCOCCNC(=O)c2csc(C(=O)c3c[nH]c4ccccc34)n2)CCC1(C)C)=CC=CC(C)=CC(=O)O<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)CNC(=O)c2cccc(-c3ccc(N4CCN(C)CC4)c(NC(=O)c4c[nH]c(=O)cc4C(F)(F)F)c3)c2)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'COc1cc2nc(N3CCN(C(=O)c4cn(CCOCCOCC(=O)Nc5cccc6c5C(=O)N(C5CCC(=O)NC5=O)C6=O)nn4)CC3)nc(N)c2cc1OC<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Nc1ncnc2c1c(-c1ccc(Oc3ccccc3)cc1)nn2C1CCCN(C(=O)c2cn(CCOCCOCCOCCCc3cccc4c3CN(C3CCC(=O)NC3=O)C4=O)nn2)C1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)COCCCOCCOC(=O)CC2N=C(c3ccc(Cl)cc3)c3c(sc(C)c3C)-n3c(C)nnc32)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1nnc2n1-c1sc(C#Cc3cnn(CCCCCc4cccc5c4CN(C4CCC(=O)NC4=O)C5=O)c3)c(Cc3ccccc3)c1COC2<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CC(=O)c1c(C)c2cnc(Nc3ccc(N4CCN(Cc5cn(CCCCCCc6cccc7c6CN(C6CCC(=O)NC6=O)C7=O)nn5)CC4)cn3)nc2n(C2CCCC2)c1=O<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CCn1c(=O)n(CC(=O)NCCCNc2cccc3c2C(=O)N(C2CCC(=O)NC2=O)C3=O)c(=O)c2cc(C(N)=O)c(N)nc21<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CC1CCC2(C(=O)NCCCCCCCCNc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)CCC3(C)C(=CCC4C5(C)CCC(O)C(C)(C)C5CCC43C)C2C1C<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CN(C)C(=O)c1cn(C2CCCC2)c2nc(Nc3ccc(NC(=O)CCCCC#Cc4ccc5c(c4)C(=O)N(C4CCC(=O)NC4=O)C5=O)cc3)ncc12<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1c(COc2cc(OCc3cccc(C#N)c3)c(CN3CCCCC3C(=O)NCCCCCCNc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)cc2Cl)cccc1-c1ccc2c(c1)OCCO2<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCCCCCc2ccc(C#Cc3ccc(N4C(=S)N(c5ccc(C#N)c(C(F)(F)F)c5)C(=O)C4(C)C)cc3)cn2)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)COCCOCCOc2ccc(-c3ccc4ncnc(Nc5ccc(OCc6cccc(F)c6)c(Cl)c5)c4c3)cc2)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CCOc1cc2nc(CCC(C)(C)C(=O)NCCCCCCCNc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)n(C)c2cc1NC(=O)c1cccc(C(F)(F)F)n1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'O=C1CCC(N2C(=O)c3cccc(NC(=O)CCCc4cn(CCC(=O)Nc5ccc(N6Cc7ccc(O)cc7OC6=O)cc5)nn4)c3C2=O)C(=O)N1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CC1(C)C(NC(=O)c2ccc(N3CCN(CC4CCN(c5ccc6c(c5)C(=O)N(C5CCC(=O)NC5=O)C6=O)CC4)CC3)cc2)C(C)(C)C1Oc1ccc(C#N)c(F)c1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCCCCCOc2ccc(C=NNC(=O)c3cccc(Oc4c(-c5ccc(O)cc5)sc5cc(O)ccc45)c3)cc2)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CCOc1cc(C(C)(C)C)ccc1C1=NC(c2ccc(Cl)cc2)C(c2ccc(Cl)cc2)N1C(=O)N1CCN(C(=O)CNC(=O)CCC#Cc2cccc3c2CN(C2CCC(=O)NC2=O)C3=O)CC1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'O=C(CCCCCCC(=O)NN=Cc1ccc(OCCCC#Cc2cccc3c2CN(C2CCC(=O)NC2=O)C3=O)cc1)NO<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1nnc2n1-c1sc(C#Cc3ccn(CCCCNc4cccc5c4CN(C4CCC(=O)NC4=O)C5=O)c3)c(Cc3ccccc3)c1COC2<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'NC(=O)c1cc2c(=O)n(CC(=O)NCCCNc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)c(=O)n(C3CC3)c2nc1N<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'COc1cc(Nc2ncc3c(n2)-c2ccc(Cl)cc2C(c2c(F)cccc2OC)=NC3)ccc1C(=O)NCCOCCOCCOCCOCCC(=O)NC(C(=O)N1CC(O)CC1C(=O)NCc1ccc(-c2scnc2C)cc1)C(C)(C)C<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'COc1cc(-c2cn(C)c(=O)c3cnccc23)cc(OC)c1CN1CC(NC(=O)CCCCOc2cc(-c3scnc3C)ccc2CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)C2(F)CC2)C(C)(C)C)C1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CNC(C)C(=O)NC(C(=O)N1Cc2cc(OCCOCCOCCOCCOCCOC(=O)N3CCCC(n4nc(-c5ccc(Oc6ccc(F)cc6F)cc5)c(C(N)=O)c4N)C3)ccc2CC1C(=O)NC1CCCc2ccccc21)C(C)(C)C<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Nc1nc(NCCNC(=O)CCCNc2cccc3c2CN(C2CCC(=O)NC2=O)C3=O)nn1-c1cc2c(nn1)-c1ccccc1CCC2<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C(=O)CCCc3cn(CCOCCOCCOCCOCCNc4cccc5c4C(=O)N(C4CCC(=O)NC4=O)C5=O)nn3)CC2)n1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C(=O)CCCCCCCCC(=O)NC(C(=O)N3CC(O)CC3C(=O)NCc3ccc(-c4scnc4C)cc3)C(C)(C)C)CC2)n1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CNC(C)C(=O)NC(C(=O)N1CCCC1c1nc(C(=O)c2ccc(F)cc2)cs1)C1CCN(C(=O)c2cnc(N3CCN(CCCOc4cc5ncnc(Nc6n[nH]c(C)c6C)c5cc4S(=O)(=O)C(C)(C)C)CC3)cn2)CC1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CC1(C)C(=O)N(c2ccc(C#N)c(C(F)(F)F)c2)C(=S)N1c1ccc(-c2ccc(OCCCCCCCCCCNc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)cc2)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CCn1c(-c2nonc2N)nc2c(C#CC(C)(C)O)ncc(OCCCNCCCC(=O)NCCOCCC(=O)NC(C(=O)N3CC(O)CC3C(=O)NCc3ccc(-c4scnc4C)cc3)C(C)(C)C)c21<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)COCC(F)(F)COc2ccc(-c3ccc(N4C(=S)N(c5ccc(C#N)c(C(F)(F)F)c5)C(=O)C4(C)C)cc3)cc2)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(C(C)NC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCCCCCCN(CCOc2ccc(C(=O)c3c(-c4ccc(O)cc4)sc4cc(O)ccc34)cc2)C2CCCC2)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)COCCOCCOCCOCCOCCOc2cc(-c3scnc3C)ccc2CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)C2(C#N)CC2)C(C)(C)C)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'NC(=O)c1c(-c2ccc(Oc3ccccc3)cc2)nn2c1NCCC2C1CCN(C(=O)CCc2ccc(N3CCN(c4ccc5c(c4)C(=O)N(C4CCC(=O)NC4=O)C5=O)CC3)cc2)CC1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ccc(C(=O)NCCCCc2cn(CCCC(=O)NCCOCCOCCNc3cccc4c3C(=O)N(C3CCC(=O)N(Cc5ccccc5[N+](=O)[O-])C3=O)C4=O)nn2)cc1-n1c(C)cc(OCc2ccc(F)cc2F)c(Br)c1=O<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CC(C)c1cnn2c(NCc3ccc(NC(=O)CCCCCCc4cccc5c4CN(C4CCC(=O)NC4=O)C5=O)cc3)nc(OC3CCN(C)CC3)nc12<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CCc1cc2c(cc1N1CCC(N3CCN(CC(=O)NCCCCCCNc4cccc5c4C(=O)N(C4CCC(=O)NC4=O)C5=O)CC3)CC1)C(C)(C)c1[nH]c3cc(C#N)ccc3c1C2=O<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Nc1nc2c(ncn2C2OC(COP(=O)(O)NCCCCCC(=O)Nc3cccc4c3CN(C3CCC(=O)NC3=O)C4=O)C(O)C2O)c(=O)[nH]1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)COCCOCCOCCOCCNC(=O)CC2N=C(c3ccc(Cl)cc3)c3c(sc(C)c3C)-n3c(C)nnc32)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'N#CC1(CNc2cccc(-c3cc(NC4CCC(NCC(=O)NCCOCCOCCOCCNc5cccc6c5C(=O)N(C5CCC(=O)NC5=O)C6=O)CC4)ncc3Cl)n2)CCOCC1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CCOC(=O)c1[nH]c2cc(Cl)ccc2c1C(C(=O)NC(C)(C)C)N(Cc1cc(F)c(F)c(F)c1)C(=O)CCCCCC#Cc1cccc2c1CN(C1CCC(=O)NC1=O)C2=O<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'O=C1CCC(N2Cc3cc(N4CCN(CC5CCN(c6ccc(C7c8ccc(O)cc8CCC7c7ccccc7)cc6)CC5)CC4)ccc3C2=O)C(=O)N1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1[nH]c(C=C2C(=O)Nc3ccc(F)cc32)c(C)c1C(=O)NCCOCCNc1ccc2c(c1)C(=O)N(C1CCC(=O)NC1=O)C2=O<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'COc1cc(-c2cn(C)c(=O)c3cnccc23)cc(OC)c1CN1CCN(C(=O)COCCOCCOCC(=O)NC(C(=O)N2CC(O)CC2C(=O)NCc2ccc(-c3scnc3C)cc2)C(C)(C)C)CC1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CNC(C)C(=O)NC1CN(C(=O)c2cnc(N3CCN(CCCOc4cc5ncnc(Nc6n[nH]c(C)c6C)c5cc4S(=O)(=O)C(C)(C)C)CC3)cn2)CCC2CCC(C(=O)NC3CCCc4ccccc43)N2C1=O<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'O=C(CCCCCNc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O)NCCCCCCNC(=O)CCCCC(=O)NCCN1C(=O)c2cccc3c(Sc4ccc(Br)cc4)ccc(c23)C1=O<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CNC(C)C(=O)NC(C(=O)N1CC(NC(=O)CCCCCCCCCN2CCN(CCCOc3ccc(-c4cnc5cccc(-c6cc(F)c(CN7CCS(=O)(=O)CC7)c(F)c6)c5n4)cc3OC)CC2)CC1C(=O)NC1CCCc2ccccc21)C1CCCCC1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'COc1cc(-c2cnc3cccc(-c4cc(F)c(CN5CCS(=O)(=O)CC5)c(F)c4)c3n2)ccc1OCCCN1CCN(CCOCCOCCOCCOCC(=O)Nc2cccc3c2CN(C2CCC(=O)NC2=O)C3=O)CC1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)COCC(=O)N2CC(Nc3cc(C(=O)NCC(O)CN4CCc5ccccc5C4)ncn3)C2)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCN(CCC(=O)NCCCCCCCCOc2cccc3c2C(=O)N(C2CCC(=O)NC2=O)C3=O)CC1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1ncsc1-c1ccc(CNC(=O)C2CC(O)CN2C(=O)C(NC(=O)CCCCCCNCCCONC(=O)c2ccc(F)c(F)c2Nc2ccc(I)cc2F)C(C)(C)C)cc1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CC1CCC2(CCN(c3cnc(Sc4cccc(NC(=O)CCC(=O)NCCCCCCCCCNc5cccc6c5C(=O)N(C5CCC(=O)NC5=O)C6=O)c4Cl)c(N)n3)CC2)C1N<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CNC(C)C(=O)NC(C(=O)N1CC(NC(=O)CCOCCOCCOCCNC(=O)CN2CCN(c3ccc(Nc4cc5c(cn4)c(C)c(C(C)=O)c(=O)n5C4CCCC4)nc3)CC2)CC1C(=O)Nc1c(F)cccc1F)C(C)(C)C<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CCn1c(-c2nonc2N)nc2c(C#CC(C)(C)O)ncc(OCCCNCCCC(=O)NCCOCCOCCOCCOCCNc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)c21<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Nc1nc(Nc2ccc(CNC(=O)CCCNc3cccc4c3CN(C3CCC(=O)NC3=O)C4=O)cc2)nn1-c1ccc(-c2ccccc2)nn1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'C=CC(=O)Nc1cccc(-n2c(=O)cc(C)c3cnc(Nc4ccc(N5CCN(C(=O)CCCCCCCCCCC(=O)Nc6cccc7c6C(=O)N(C6CCC(=O)NC6=O)C7=O)CC5)cc4OC)nc32)c1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'COc1cc(OCc2ccc(-c3ccccc3)c(C)c2)cc(OC)c1CNCC(=O)NCCCNC(=O)COc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'CCCS(=O)(=O)Nc1ccc(F)c(-n2cc(-c3cncnc3)c3nc(N(C)C4CCN(C(=O)CCCCNc5cccc6c5C(=O)N(C5CCC(=O)NC5=O)C6=O)CC4)ccc32)c1F<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'O=C1CCC(N2Cc3c(NC(=O)COCCOCCOCCOCCNCCNc4nonc4C(=NO)Nc4ccc(F)c(Br)c4)cccc3C2=O)C(=O)N1<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>', 'Cc1[nH]c(C=C2C(=O)Nc3ccc(S(=O)(=O)Cc4c(Cl)cccc4Cl)cc32)c(C)c1C(=O)NCCCCCNc1ccc2c(c1)C(=O)N(C1CCC(=O)NC1=O)C2=O<s><s>[*:1]c1ccc(Nc2ncc3cc(NC(=O)C4CC5CCC(C)(C)C)n3)cc2)cc1.[*:2]Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O.[*:2]C(=O)NCCOCCOCC[*:1]</s>']"""
texts = eval(texts)
queries = [s.split("<s><s>")[0].replace("<s>", "") for s in texts]
responses = [s.split("<s><s>")[-1].replace("</s>", "") for s in texts]

In [1]:
!pip install datasets transformers peft accelerate bitsandbytes evaluate rouge_score huggingface_hub rdkit tensorboard -qqq -U

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 493.7/493.7 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 22.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 18.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.4/261.4 kB 22.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.6/92.6 MB 10.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 36.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.5/30.5 MB 53.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 105.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 14.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 16.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 105

In [2]:
!git clone https://github.com/huggingface/trl.git
%cd trl
!pip install . -qqq
%cd ..

Cloning into 'trl'...
remote: Enumerating objects: 4800, done.
remote: Counting objects: 100% (1669/1669), done.
remote: Compressing objects: 100% (430/430), done.
remote: Total 4800 (delta 1503), reused 1248 (delta 1239), pack-reused 3131
Receiving objects: 100% (4800/4800), 5.82 MiB | 21.50 MiB/s, done.
Resolving deltas: 100% (3053/3053), done.
/content/trl
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.9/99.9 kB 2.0 MB/s eta 0:00:00
/content


In [2]:
!git config --global user.name "ribesstefano"
!git config --global user.mail "ribes.stefano@gmail.com"
!git clone https://github.com/ribesstefano/trl.git
%cd trl
!git checkout encoder_decoder_patch
!pip install . -qqq

Cloning into 'trl'...
remote: Enumerating objects: 3508, done.
remote: Counting objects: 100% (3507/3507), done.
remote: Compressing objects: 100% (1349/1349), done.
remote: Total 3508 (delta 2222), reused 3106 (delta 1969), pack-reused 1
Receiving objects: 100% (3508/3508), 5.40 MiB | 18.44 MiB/s, done.
Resolving deltas: 100% (2222/2222), done.
/content/trl
Branch 'encoder_decoder_patch' set up to track remote branch 'encoder_decoder_patch' from 'origin'.
Switched to a new branch 'encoder_decoder_patch'
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.9/99.9 kB 749.5 kB/s eta 0:00:00


In [19]:
%cd ..
!ls

/content
sample_data  trl


## TRL

In [3]:
from huggingface_hub import notebook_login

notebook_login()

In [4]:
import torch
from transformers import (
    EncoderDecoderModel,
    EncoderDecoderConfig,
    AutoTokenizer,
)
from trl import (
    AutoModelForCausalLMWithValueHead,
    AutoModelForSeq2SeqLMWithValueHead,
    PPOConfig,
    PPOTrainer,
    create_reference_model,
)

/usr/local/lib/python3.10/dist-packages/trl/trainer/ppo_config.py:141: UserWarning: The `optimize_cuda_cache` arguement will be deprecated soon, please use `optimize_device_cache` instead.
  warnings.warn(


### Setup Model

In [6]:
pretrained_model = "ailab-bio/PROTAC-Splitter_untied_80-20-split"
# tokenizer = AutoTokenizer.from_pretrained(pretrained_model)
# bert2bert = EncoderDecoderModel.from_pretrained(pretrained_model)
config = EncoderDecoderConfig.from_pretrained(pretrained_model)

In [7]:
from transformers import AutoTokenizer
from trl import AutoModelForSeq2SeqLMWithValueHead, PPOConfig, PPOTrainer

model = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(pretrained_model)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model)

tokenizer.pad_token = tokenizer.eos_token

### Setup Dataset

In [53]:
from datasets import concatenate_datasets, load_dataset

train_dataset = load_dataset("ailab-bio/PROTAC-Substructures", "80-20-split", split="train")
train_dataset = train_dataset.rename_column("text", "query")
train_dataset = train_dataset.remove_columns(["labels"])

unlabeled_dataset = load_dataset("ailab-bio/PROTAC-Substructures", "unlabeled", split="train")
unlabeled_dataset = unlabeled_dataset.rename_column("text", "query")
unlabeled_dataset = unlabeled_dataset.remove_columns(["labels"])

dataset = concatenate_datasets([train_dataset, unlabeled_dataset])
dataset

Dataset({
    features: ['query'],
    num_rows: 3239
})

In [54]:
def tokenize(sample):
    sample["input_ids"] = tokenizer.encode(sample["query"], padding='max_length')
    return sample

train_dataset = dataset.map(tokenize, batched=False)
train_dataset

Map:   0%|          | 0/3239 [00:00<?, ? examples/s]

Dataset({
    features: ['query', 'input_ids'],
    num_rows: 3239
})

## Reward Function

In [55]:
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs, MACCSkeys, rdFMCS, Draw
from rdkit import RDLogger

def is_valid_smiles(smiles: str) -> bool:
    mol = Chem.MolFromSmiles(smiles)
    return True if mol is not None else False

def has_three_substructures(smiles: str) -> bool:
    return smiles.count(".") == 2

def has_all_attachment_points(smiles: str) -> bool:
    return smiles.count("[*:1]") == 2 and smiles.count("[*:2]") == 2

def is_substructure(protac_smiles, substruct_smiles) -> bool:
    protac_mol = Chem.MolFromSmiles(protac_smiles)
    substruct_mol = Chem.MolFromSmarts(substruct_smiles)
    return protac_mol.HasSubstructMatch(substruct_mol)

def same_atom_counts_and_types(smiles1, smiles2, get_atoms_diff=False):
    """
    Check if two molecules have the same number and types of atoms.

    Args:
    smiles1 (str): SMILES notation for the first molecule.
    smiles2 (str): SMILES notation for the second molecule.

    Returns:
    bool: True if the molecules have the same atom counts and types, False otherwise.
    """
    mol1 = Chem.MolFromSmiles(smiles1)
    mol2 = Chem.MolFromSmiles(smiles2)
    if mol1 is None or mol2 is None:
        if get_atoms_diff:
            return float("nan")
            # raise ValueError("Invalid SMILES notation provided for one or both molecules.")
        else:
            return False
    num_atoms1 = Chem.rdMolDescriptors.CalcNumHeavyAtoms(mol1)
    num_atoms2 = Chem.rdMolDescriptors.CalcNumHeavyAtoms(mol2)
    if get_atoms_diff:
        return abs(num_atoms1 - num_atoms2)
        # tmp = {}
        # for atom in atom_counts1.keys():
        #     tmp[atom] = int(abs(atom_counts1.get(atom, 0) - atom_counts2.get(atom, 0)))
        # for atom in atom_counts2.keys():
        #     tmp[atom] = int(abs(atom_counts1.get(atom, 0) - atom_counts2.get(atom, 0)))
        # return tmp # abs(atom_counts1.get('O', 0) - atom_counts2.get('O', 0))
    else:
        atom_counts1, atom_counts2 = {}, {}
        for atom in mol1.GetAtoms():
            if '*' not in atom.GetSmarts():
                atom_counts1[atom.GetSymbol()] = atom_counts1.get(atom.GetSymbol(), 0) + 1
        for atom in mol2.GetAtoms():
            if '*' not in atom.GetSmarts():
                atom_counts2[atom.GetSymbol()] = atom_counts2.get(atom.GetSymbol(), 0) + 1
        return (atom_counts1 == atom_counts2) & (num_atoms1 == num_atoms2)

RDLogger.DisableLog("rdApp.*")

In [43]:
print(is_valid_smiles(responses[0]))
print(has_three_substructures(responses[0]))
print(has_all_attachment_points(responses[0]))
print(same_atom_counts_and_types(responses[0], queries[0], get_atoms_diff=False))
print(is_substructure(queries[0], responses[0].split(".")[0]))

False
True
True
False
False


In [58]:
def reward_function(query, response) -> float:
    if not is_valid_smiles(response):
        return -50.
    if not has_three_substructures(response):
        return -50.
    if not has_all_attachment_points(response):
        return -50.
    # if not same_atom_counts_and_types(response, query, get_atoms_diff=False):
    #     return 0.
    return 1. - same_atom_counts_and_types(response, query, get_atoms_diff=True)
    # substructures = response.split(".")
    # for substructure in substructures:
    #     if not is_substructure(query, response):
    #         return 0.
    # return 1.

for i in range(len(responses)):
    r = reward_function(queries[i], responses[i])
    print(r)
    if i > 10:
        break

-50.0
-50.0
-50.0
-50.0
-50.0
-50.0
-50.0
-50.0
-50.0
-24.0
-50.0
-50.0


## Training Loop

In [57]:
from trl import PPOConfig, PPOTrainer

ppo_config = PPOConfig(
    model_name=pretrained_model,
    learning_rate=1.41e-5,
    seed=42,
    steps=2000, # Default: 20_000
    ppo_epochs=4, # Default: 4
    is_encoder_decoder=True,
    global_batch_size=32,
)

ppo_trainer = PPOTrainer(
    model=model,
    config=ppo_config,
    tokenizer=tokenizer,
    dataset=train_dataset,
)

In [63]:
from tqdm import tqdm

generation_kwargs = {
    "do_sample": False,
    "pad_token_id": tokenizer.eos_token_id,
}

def clean_text(text: str) -> str:
    return text.replace("<s>", "").replace("</s>", "")

for epoch, batch in tqdm(enumerate(ppo_trainer.dataloader)):
    query_tensors = batch["input_ids"]

    #### Get response from SFTModel
    response_tensors = ppo_trainer.generate(query_tensors, **generation_kwargs)
    batch["response"] = [tokenizer.decode(r.squeeze()) for r in response_tensors]

    #### Compute reward score
    rewards = [reward_function(clean_text(q), clean_text(r)) for q, r in zip(batch["query"], batch["response"])]
    print(f"rewards: {rewards}")
    rewards = [torch.tensor(r) for r in rewards]
    print(f"rewards: {rewards}")

    print(f"len(query_tensors): {len(query_tensors)}")
    print(f"len(response_tensors): {len(response_tensors)}")
    print(f"len(rewards): {len(rewards)}")

    #### Run PPO step
    stats = ppo_trainer.step(query_tensors, response_tensors, rewards)
    ppo_trainer.log_stats(stats, batch, rewards)

#### Save model
ppo_trainer.save_model(r"/content/drive/MyDrive/Colab Notebooks/PROTAC-Splitter_PPO")

0it [02:17, ?it/s]

rewards: [-50.0, -5.0, -40.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, 1.0, -50.0, 0.0, -5.0, -50.0, -50.0, -8.0, -21.0, -40.0, -24.0, -50.0, -34.0, -41.0, -50.0, -50.0, -14.0, -50.0, -16.0, -50.0, -23.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -7.0, -16.0, -50.0, -50.0, -7.0, -50.0, -50.0, -50.0, -32.0, -5.0, -12.0, -24.0, -50.0, -18.0, -27.0, -50.0, -50.0, -1.0, -50.0, -4.0, -2.0, -50.0, -41.0, -15.0, -50.0, -50.0, -2.0, -14.0, -50.0, -13.0, -32.0, -25.0, -50.0, -10.0, -9.0, -50.0, -50.0, -50.0, -50.0, -9.0, -9.0, -50.0, -2.0, -4.0, -50.0, -50.0, -50.0, -50.0, -6.0, -5.0, -50.0, -9.0, -50.0, -50.0, -8.0, -26.0, -50.0, -50.0, 0.0, -50.0, -50.0, -1.0, -16.0, -50.0, -15.0, -50.0, -11.0, -50.0, -50.0, -14.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -

ValueError: ignored

In [3]:
import torch
from transformers import (
    EncoderDecoderModel,
    EncoderDecoderConfig,
    AutoTokenizer,
)
from trl import (
    AutoModelForCausalLMWithValueHead,
    AutoModelForSeq2SeqLMWithValueHead,
    PPOConfig,
    PPOTrainer,
    create_reference_model,
)

pretrained_model = "ribesstefano/ChemBERTa2ChemBERTa-58M"
# tokenizer = AutoTokenizer.from_pretrained(pretrained_model)
bert2bert = EncoderDecoderModel.from_pretrained(pretrained_model)
config = EncoderDecoderConfig.from_pretrained(pretrained_model)
# print(bert2bert.encoder.config.hidden_size)
# print(bert2bert.decoder.config.hidden_size)
# print(bert2bert.config.hidden_size)

model = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(pretrained_model)
# # model_ref = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(pretrained_model, device_map="auto", load_in_8bit=True)
# model_ref = create_reference_model(model, num_shared_layers=6)

# tokenizer = AutoTokenizer.from_pretrained(pretrained_model)
# tokenizer.pad_token = tokenizer.eos_token

/content/trl/trl/trainer/ppo_config.py:141: UserWarning: The `optimize_cuda_cache` arguement will be deprecated soon, please use `optimize_device_cache` instead.
  warnings.warn(


The following encoder weights were not tied to the decoder ['roberta/pooler']
The following encoder weights were not tied to the decoder ['roberta/pooler']
The following encoder weights were not tied to the decoder ['roberta/pooler']
The following encoder weights were not tied to the decoder ['roberta/pooler']


The following encoder weights were not tied to the decoder ['roberta/pooler']
The following encoder weights were not tied to the decoder ['roberta/pooler']
The following encoder weights were not tied to the decoder ['roberta/pooler']
The following encoder weights were not tied to the decoder ['roberta/pooler']
